# PRISM Complete Experiment

**Platform-Robust In-Field Sequential Monitoring for Silicon Lifecycle
Management**

This notebook is the canonical executable source for the PRISM experiment. It
contains the collection package, workload harness, command-line operations,
validation, unit tests, and guarded experiment controls that previously lived
in separate Python files.

Run the notebook from top to bottom once on each machine. The source cells
materialize an ephemeral `.prism_runtime/` tree because the collection engine
starts isolated subprocesses. That tree is Git-ignored and disposable; rerun
the source cells to rebuild it. Raw data continue to be written to the
repository's Git-ignored `data/raw/` directory.

Safety rules:

- `Run All` performs setup, preflight, and unit tests only.
- Hardware probes, DICE import, smoke collection, and production collection
  remain disabled until their explicit boolean guard is changed to `True`.
- Production collection still requires a clean committed Git revision.
- Do not edit `.prism_runtime/`; edit the corresponding cell in this notebook.


## Experiment flow

1. Materialize and validate the complete runtime.
2. Optionally import the frozen DICE baseline for retrospective comparison.
3. Probe Apple M2 Pro capabilities and collect a matched smoke pair.
4. Repeat the same probe and smoke gate on AMD EPYC/Linux.
5. Freeze the shared schema after both smoke gates pass.
6. Collect calibration/development rows, then freeze the method.
7. Explicitly unlock the held-out test rows.
8. Monitor run validity, channel availability, and benign hours daily.

The immutable plan declares 252 runs and 64.8 total machine-hours across the
two platforms. Review `docs/extension-collection-contract.md` before production.


## 1. Locate the repository and create the disposable runtime


In [ ]:

from __future__ import annotations

import os
import shutil
import subprocess
import sys
from pathlib import Path


def find_repository_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (
            (candidate / ".git").exists()
            and (candidate / "configs" / "experiment-matrix.toml").is_file()
        ):
            return candidate
    raise RuntimeError(
        "Open this notebook from inside the PRISM repository; "
        "the repository root could not be located."
    )


REPO_ROOT = find_repository_root(Path.cwd())
RUNTIME_ROOT = REPO_ROOT / ".prism_runtime"
RUNTIME_ROOT.mkdir(parents=True, exist_ok=True)
for runtime_directory in ("src/prism_slm", "scripts", "tests"):
    (RUNTIME_ROOT / runtime_directory).mkdir(parents=True, exist_ok=True)
os.chdir(REPO_ROOT)
os.environ["PRISM_REPO_ROOT"] = str(REPO_ROOT)

runtime_src = str(RUNTIME_ROOT / "src")
existing_pythonpath = os.environ.get("PYTHONPATH", "")
os.environ["PYTHONPATH"] = (
    runtime_src
    if not existing_pythonpath
    else runtime_src + os.pathsep + existing_pythonpath
)

print(f"Repository: {REPO_ROOT}")
print(f"Disposable runtime: {RUNTIME_ROOT}")


## 2. Embedded source

Each cell below is a complete former Python file. Running the cell writes that
source into `.prism_runtime/` without changing tracked repository files.


### `src/prism_slm/__init__.py`


In [ ]:
%%writefile .prism_runtime/src/prism_slm/__init__.py
"""PRISM-SLM research scaffolding."""

from .contracts import ChannelSpec, PlatformSpec, SemanticGroup

__all__ = ["ChannelSpec", "PlatformSpec", "SemanticGroup"]


### `src/prism_slm/contracts.py`


In [ ]:
%%writefile .prism_runtime/src/prism_slm/contracts.py
"""Typed contracts for heterogeneous host telemetry."""

from __future__ import annotations

from dataclasses import dataclass, field
from enum import Enum
from typing import Iterable


class SemanticGroup(str, Enum):
    COMPUTE = "compute"
    MEMORY = "memory"
    IO = "io"
    THERMAL_POWER = "thermal_power"
    ACCELERATOR = "accelerator"
    AVAILABILITY = "availability"


@dataclass(frozen=True)
class ChannelSpec:
    native_name: str
    unit: str
    semantic_group: SemanticGroup
    source: str
    sampling_period_seconds: float
    cumulative: bool = False
    description: str = ""

    def __post_init__(self) -> None:
        if not self.native_name.strip():
            raise ValueError("native_name must be non-empty")
        if not self.unit.strip():
            raise ValueError("unit must be non-empty")
        if not self.source.strip():
            raise ValueError("source must be non-empty")
        if self.sampling_period_seconds <= 0:
            raise ValueError("sampling_period_seconds must be positive")


@dataclass(frozen=True)
class PlatformSpec:
    platform_id: str
    architecture: str
    operating_system: str
    observability: str
    channels: tuple[ChannelSpec, ...] = field(default_factory=tuple)

    @classmethod
    def from_channels(
        cls,
        *,
        platform_id: str,
        architecture: str,
        operating_system: str,
        observability: str,
        channels: Iterable[ChannelSpec],
    ) -> "PlatformSpec":
        return cls(
            platform_id=platform_id,
            architecture=architecture,
            operating_system=operating_system,
            observability=observability,
            channels=tuple(channels),
        )

    def __post_init__(self) -> None:
        values = {
            "platform_id": self.platform_id,
            "architecture": self.architecture,
            "operating_system": self.operating_system,
            "observability": self.observability,
        }
        for name, value in values.items():
            if not value.strip():
                raise ValueError(f"{name} must be non-empty")

        names = [channel.native_name for channel in self.channels]
        if len(names) != len(set(names)):
            raise ValueError("channel native_name values must be unique per platform")

    @property
    def available_groups(self) -> frozenset[SemanticGroup]:
        return frozenset(channel.semantic_group for channel in self.channels)


### `src/prism_slm/workload_harness.py`


In [ ]:
%%writefile .prism_runtime/src/prism_slm/workload_harness.py
"""Cross-platform workload and pressure harness for PRISM collection.

The harness is intentionally a separate process. Terminating a collector does
not leave a compute loop running, and every independent run starts fresh state.
"""

from __future__ import annotations

import argparse
import hashlib
import os
import random
import signal
import threading
import time
import urllib.parse
import zlib

import numpy as np


SEED = 830616
WORKLOADS = ("PY_STATS", "PY_AI", "BROWSER", "VIDEO_SW")
STRESSORS = (
    "NOMINAL",
    "ATOMIC",
    "BRANCH",
    "CACHE",
    "MEMBW",
    "TLB",
    "THERMAL_SHIFT",
    "POWER_SHIFT",
    "DEGRADATION_PROXY",
)
CONTROL_SCENARIOS = ("CONTROLLED_CRASH", "TELEMETRY_INTERRUPTION")
SCENARIOS = STRESSORS + CONTROL_SCENARIOS


def _stop_event() -> threading.Event:
    event = threading.Event()

    def stop(_signum: int, _frame: object) -> None:
        event.set()

    signal.signal(signal.SIGTERM, stop)
    signal.signal(signal.SIGINT, stop)
    return event


def _memory_mb() -> int:
    value = int(os.environ.get("PRISM_STRESS_MB", "128"))
    return min(max(value, 16), 1024)


def workload_py_stats(stop: threading.Event) -> None:
    rng = np.random.default_rng(SEED)
    values = rng.standard_normal(8_000_000, dtype=np.float32)
    while not stop.is_set():
        _ = (
            float(values.mean()),
            float(values.std()),
            float(np.percentile(values[::16], 95)),
        )
        values[::1024] += np.float32(0.0001)


def workload_py_ai(stop: threading.Event) -> None:
    rng = np.random.default_rng(SEED)
    left = rng.standard_normal((768, 768), dtype=np.float32)
    right = rng.standard_normal((768, 768), dtype=np.float32)
    while not stop.is_set():
        result = left @ right
        left, right = right, np.tanh(result).astype(np.float32, copy=False)


def workload_browser(stop: threading.Event) -> None:
    # A network-free, deterministic web-processing workload. It avoids making
    # Internet availability a hidden experimental variable.
    blocks = [
        (
            f"<article id='{idx}'><a href='/paper/{idx}?view=full'>"
            f"Telemetry article {idx}</a><p>{'silicon ' * 200}</p></article>"
        )
        for idx in range(512)
    ]
    document = ("<html><body>" + "".join(blocks) + "</body></html>").encode()
    while not stop.is_set():
        compressed = zlib.compress(document, level=3)
        restored = zlib.decompress(compressed).decode()
        for token in restored.split("href='")[1::32]:
            urllib.parse.urlparse(token.split("'", 1)[0])
        hashlib.sha256(compressed).digest()


def workload_video_sw(stop: threading.Event) -> None:
    rng = np.random.default_rng(SEED)
    frame = rng.integers(0, 256, size=(720, 1280, 3), dtype=np.uint8)
    while not stop.is_set():
        gray = (
            0.299 * frame[:, :, 0]
            + 0.587 * frame[:, :, 1]
            + 0.114 * frame[:, :, 2]
        ).astype(np.uint8)
        compressed = zlib.compress(gray[::2, ::2].tobytes(), level=1)
        zlib.decompress(compressed)
        frame = np.roll(frame, 1, axis=1)


def stress_nominal(stop: threading.Event) -> None:
    stop.wait()


def stress_atomic(stop: threading.Event) -> None:
    lock = threading.Lock()
    counter = [0]

    def worker() -> None:
        while not stop.is_set():
            with lock:
                counter[0] += 1

    count = max(2, (os.cpu_count() or 2) // 2)
    threads = [threading.Thread(target=worker, daemon=True) for _ in range(count)]
    for thread in threads:
        thread.start()
    while not stop.wait(0.1):
        pass


def stress_branch(stop: threading.Event) -> None:
    rng = random.Random(SEED)
    value = 0
    while not stop.is_set():
        bits = rng.getrandbits(32)
        if bits & 1:
            value += 3
        elif bits & 2:
            value -= 2
        elif bits & 4:
            value ^= bits
        else:
            value += 1


def stress_cache(stop: threading.Event) -> None:
    rng = np.random.default_rng(SEED)
    count = _memory_mb() * 1024 * 1024 // np.dtype(np.float32).itemsize
    values = rng.standard_normal(count, dtype=np.float32)
    indices = rng.integers(0, count, size=min(count, 1_000_000), dtype=np.int64)
    while not stop.is_set():
        _ = float(values[indices].sum())
        indices = np.roll(indices, 1024)


def stress_membw(stop: threading.Event) -> None:
    size = _memory_mb() * 1024 * 1024
    source = np.zeros(size, dtype=np.uint8)
    target = np.ones(size, dtype=np.uint8)
    while not stop.is_set():
        np.copyto(source, target)
        source += np.uint8(1)
        source, target = target, source


def stress_tlb(stop: threading.Event) -> None:
    rng = np.random.default_rng(SEED)
    size = _memory_mb() * 1024 * 1024
    values = np.zeros(size, dtype=np.uint8)
    pages = np.arange(0, size, 4096, dtype=np.int64)
    while not stop.is_set():
        rng.shuffle(pages)
        for start in range(0, len(pages), 4096):
            if stop.is_set():
                return
            block = pages[start : start + 4096]
            values[block] = values[block] + np.uint8(1)


def _matrix_burst(
    stop: threading.Event,
    *,
    active_seconds: float,
    idle_seconds: float,
) -> None:
    rng = np.random.default_rng(SEED)
    left = rng.standard_normal((512, 512), dtype=np.float32)
    right = rng.standard_normal((512, 512), dtype=np.float32)
    while not stop.is_set():
        deadline = time.monotonic() + active_seconds
        while time.monotonic() < deadline and not stop.is_set():
            result = left @ right
            left, right = right, np.tanh(result).astype(np.float32, copy=False)
        if idle_seconds > 0:
            stop.wait(idle_seconds)


def stress_thermal_shift(stop: threading.Event) -> None:
    """Sustained compute-load change used as a portable thermal-condition proxy."""

    _matrix_burst(stop, active_seconds=2.0, idle_seconds=0.02)


def stress_power_shift(stop: threading.Event) -> None:
    """Repeatable duty-cycled compute used as a portable power-demand shift."""

    _matrix_burst(stop, active_seconds=0.35, idle_seconds=0.15)


def stress_degradation_proxy(stop: threading.Event) -> None:
    """Progressively increase compute duty cycle without claiming physical aging."""

    rng = np.random.default_rng(SEED)
    left = rng.standard_normal((512, 512), dtype=np.float32)
    right = rng.standard_normal((512, 512), dtype=np.float32)
    stage = 0
    stage_started = time.monotonic()
    while not stop.is_set():
        active_seconds = min(0.10 + 0.08 * stage, 0.90)
        idle_seconds = max(1.0 - active_seconds, 0.10)
        deadline = time.monotonic() + active_seconds
        while time.monotonic() < deadline and not stop.is_set():
            result = left @ right
            left, right = right, np.tanh(result).astype(np.float32, copy=False)
        stop.wait(idle_seconds)
        if time.monotonic() - stage_started >= 30:
            stage = min(stage + 1, 10)
            stage_started = time.monotonic()


WORKLOAD_FUNCTIONS = {
    "PY_STATS": workload_py_stats,
    "PY_AI": workload_py_ai,
    "BROWSER": workload_browser,
    "VIDEO_SW": workload_video_sw,
}
STRESSOR_FUNCTIONS = {
    "NOMINAL": stress_nominal,
    "ATOMIC": stress_atomic,
    "BRANCH": stress_branch,
    "CACHE": stress_cache,
    "MEMBW": stress_membw,
    "TLB": stress_tlb,
    "THERMAL_SHIFT": stress_thermal_shift,
    "POWER_SHIFT": stress_power_shift,
    "DEGRADATION_PROXY": stress_degradation_proxy,
}


def main() -> int:
    parser = argparse.ArgumentParser()
    parser.add_argument("--role", choices=("workload", "stressor"), required=True)
    parser.add_argument("--name", required=True)
    args = parser.parse_args()

    name = args.name.upper()
    functions = WORKLOAD_FUNCTIONS if args.role == "workload" else STRESSOR_FUNCTIONS
    if name not in functions:
        raise SystemExit(f"Unsupported {args.role}: {name}")

    stop = _stop_event()
    functions[name](stop)
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


### `src/prism_slm/collection.py`


In [ ]:
%%writefile .prism_runtime/src/prism_slm/collection.py
"""Synchronized, provenance-safe telemetry collection for PRISM."""

from __future__ import annotations

import csv
import hashlib
import json
import math
import os
import platform
import re
import shutil
import statistics
import subprocess
import sys
import threading
import time
import traceback
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Iterable

import psutil

from .contracts import SemanticGroup
from .workload_harness import SCENARIOS, STRESSORS, WORKLOADS


QUALITY_CONTRACT = {
    "primary_channel_minimum_fraction": 0.95,
    "apple_temperature_minimum_fraction": 0.70,
    "apple_temperature_min_celsius": 15.0,
    "apple_temperature_max_celsius": 125.0,
    "maximum_quality_flag_fraction": 0.25,
}


def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat().replace("+00:00", "Z")


def utc_date() -> str:
    return datetime.now(timezone.utc).date().isoformat()


def safe_json(value: Any) -> Any:
    if isinstance(value, float) and not math.isfinite(value):
        return None
    if isinstance(value, dict):
        return {str(key): safe_json(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [safe_json(item) for item in value]
    return value


def dump_json(path: Path, value: Any) -> None:
    path.write_text(
        json.dumps(safe_json(value), indent=2, sort_keys=True) + "\n",
        encoding="utf-8",
    )


def append_jsonl(stream: Any, value: dict[str, Any]) -> None:
    stream.write(json.dumps(safe_json(value), sort_keys=True) + "\n")
    stream.flush()


def _rate(current: float | None, previous: float | None, elapsed: float) -> float | None:
    if current is None or previous is None or elapsed <= 0:
        return None
    delta = current - previous
    return delta / elapsed if delta >= 0 else None


class HostSampler:
    """Portable host telemetry backed by psutil."""

    def __init__(self) -> None:
        psutil.cpu_percent(interval=None)
        psutil.cpu_times_percent(interval=None)
        self.previous_time: float | None = None
        self.previous_cpu_times: Any = None
        self.previous_disk: Any = None
        self.previous_net: Any = None

    def sample(self) -> dict[str, Any]:
        now = time.monotonic()
        elapsed = now - self.previous_time if self.previous_time is not None else 0.0
        output: dict[str, Any] = {}

        output["cpu_percent"] = psutil.cpu_percent(interval=None)
        output["cpu_percent_per_logical"] = psutil.cpu_percent(interval=None, percpu=True)
        try:
            cumulative_cpu = psutil.cpu_times()
        except (OSError, PermissionError):
            cumulative_cpu = None
        derived_utilization = None
        if cumulative_cpu is not None and self.previous_cpu_times is not None:
            fields = (
                "user",
                "nice",
                "system",
                "idle",
                "iowait",
                "irq",
                "softirq",
                "steal",
            )
            total_delta = sum(
                max(
                    0.0,
                    float(getattr(cumulative_cpu, field, 0.0))
                    - float(getattr(self.previous_cpu_times, field, 0.0)),
                )
                for field in fields
            )
            idle_delta = max(
                0.0,
                float(getattr(cumulative_cpu, "idle", 0.0))
                - float(getattr(self.previous_cpu_times, "idle", 0.0)),
            ) + max(
                0.0,
                float(getattr(cumulative_cpu, "iowait", 0.0))
                - float(getattr(self.previous_cpu_times, "iowait", 0.0)),
            )
            if total_delta > 0:
                derived_utilization = 100.0 * (1.0 - min(idle_delta, total_delta) / total_delta)
        output["cpu_utilization_derived_percent"] = derived_utilization
        cpu_times = psutil.cpu_times_percent(interval=None)
        output["cpu_times_percent"] = {
            name: getattr(cpu_times, name, None)
            for name in (
                "user",
                "system",
                "idle",
                "nice",
                "iowait",
                "irq",
                "softirq",
                "steal",
            )
        }
        try:
            output["load_average"] = list(psutil.getloadavg())
        except (AttributeError, OSError):
            output["load_average"] = None

        try:
            stats = psutil.cpu_stats()
        except (OSError, PermissionError):
            stats = None
        output["context_switches_total"] = getattr(stats, "ctx_switches", None)
        output["interrupts_total"] = getattr(stats, "interrupts", None)
        output["soft_interrupts_total"] = getattr(stats, "soft_interrupts", None)
        output["syscalls_total"] = getattr(stats, "syscalls", None)

        try:
            frequency = psutil.cpu_freq()
        except (OSError, PermissionError):
            frequency = None
        output["cpu_frequency_mhz"] = (
            {
                "current": frequency.current,
                "minimum": frequency.min,
                "maximum": frequency.max,
            }
            if frequency is not None
            else None
        )

        try:
            memory = psutil.virtual_memory()
        except (OSError, PermissionError):
            memory = None
        output["memory"] = (
            {
                name: getattr(memory, name, None)
                for name in (
                    "total",
                    "available",
                    "used",
                    "free",
                    "active",
                    "inactive",
                    "wired",
                    "cached",
                    "percent",
                )
            }
            if memory is not None
            else None
        )
        try:
            swap = psutil.swap_memory()
        except (OSError, PermissionError):
            swap = None
        output["swap"] = (
            {
                name: getattr(swap, name, None)
                for name in ("total", "used", "free", "percent", "sin", "sout")
            }
            if swap is not None
            else None
        )

        try:
            disk = psutil.disk_io_counters(perdisk=False)
        except (OSError, PermissionError):
            disk = None
        if disk is not None:
            output["disk"] = {
                "read_bytes_total": disk.read_bytes,
                "write_bytes_total": disk.write_bytes,
                "read_count_total": disk.read_count,
                "write_count_total": disk.write_count,
                "read_bytes_per_second": _rate(
                    disk.read_bytes,
                    self.previous_disk.read_bytes if self.previous_disk else None,
                    elapsed,
                ),
                "write_bytes_per_second": _rate(
                    disk.write_bytes,
                    self.previous_disk.write_bytes if self.previous_disk else None,
                    elapsed,
                ),
            }
        else:
            output["disk"] = None

        try:
            net = psutil.net_io_counters(pernic=False)
        except (OSError, PermissionError):
            net = None
        if net is not None:
            output["network"] = {
                "bytes_sent_total": net.bytes_sent,
                "bytes_received_total": net.bytes_recv,
                "packets_sent_total": net.packets_sent,
                "packets_received_total": net.packets_recv,
                "bytes_sent_per_second": _rate(
                    net.bytes_sent,
                    self.previous_net.bytes_sent if self.previous_net else None,
                    elapsed,
                ),
                "bytes_received_per_second": _rate(
                    net.bytes_recv,
                    self.previous_net.bytes_recv if self.previous_net else None,
                    elapsed,
                ),
            }
        else:
            output["network"] = None

        try:
            output["process_count"] = len(psutil.pids())
        except (OSError, PermissionError):
            output["process_count"] = None
        try:
            output["uptime_seconds"] = time.time() - psutil.boot_time()
        except (OSError, PermissionError):
            output["uptime_seconds"] = None
        self.previous_time = now
        self.previous_cpu_times = cumulative_cpu
        self.previous_disk = disk
        self.previous_net = net
        return safe_json(output)


class MacmonStream:
    def __init__(
        self,
        executable: str,
        interval_ms: int,
        log_path: Path,
        raw_path: Path,
    ) -> None:
        self.executable = executable
        self.interval_ms = interval_ms
        self.log_handle = log_path.open("w", encoding="utf-8")
        self.raw_handle = raw_path.open("w", encoding="utf-8")
        self.process: subprocess.Popen[str] | None = None
        self.thread: threading.Thread | None = None
        self.lock = threading.Lock()
        self.latest_sample: dict[str, Any] | None = None
        self.sample_count = 0

    def start(self, timeout_seconds: float = 10.0) -> None:
        self.process = subprocess.Popen(
            [self.executable, "pipe", "-i", str(self.interval_ms)],
            stdout=subprocess.PIPE,
            stderr=self.log_handle,
            text=True,
            bufsize=1,
        )

        def read() -> None:
            assert self.process is not None and self.process.stdout is not None
            for line in self.process.stdout:
                self.raw_handle.write(line)
                self.raw_handle.flush()
                try:
                    sample = json.loads(line)
                except json.JSONDecodeError:
                    continue
                if isinstance(sample, dict):
                    with self.lock:
                        self.latest_sample = sample
                        self.sample_count += 1

        self.thread = threading.Thread(target=read, daemon=True)
        self.thread.start()
        deadline = time.monotonic() + timeout_seconds
        while time.monotonic() < deadline:
            with self.lock:
                if self.latest_sample is not None:
                    return
            if self.process.poll() is not None:
                break
            time.sleep(0.05)
        self.stop()
        raise RuntimeError("macmon produced no JSON telemetry during its startup probe")

    def latest(self) -> dict[str, Any] | None:
        with self.lock:
            return safe_json(dict(self.latest_sample)) if self.latest_sample else None

    def stop(self) -> None:
        if self.process is not None and self.process.poll() is None:
            self.process.terminate()
            try:
                self.process.wait(timeout=5)
            except subprocess.TimeoutExpired:
                self.process.kill()
                self.process.wait(timeout=5)
        if self.thread is not None:
            self.thread.join(timeout=2)
        if not self.log_handle.closed:
            self.log_handle.close()
        if not self.raw_handle.closed:
            self.raw_handle.close()


@dataclass(frozen=True)
class SysfsChannel:
    key: str
    path: Path
    unit: str
    divisor: float


def _slug(value: str) -> str:
    return re.sub(r"[^a-z0-9]+", "_", value.lower()).strip("_") or "unknown"


class LinuxSysfsSampler:
    def __init__(self) -> None:
        self.channels = self._discover()
        self.frequency_paths = sorted(
            Path("/sys/devices/system/cpu").glob("cpu[0-9]*/cpufreq/scaling_cur_freq")
        )

    @staticmethod
    def _discover() -> list[SysfsChannel]:
        result: list[SysfsChannel] = []
        seen: set[str] = set()

        def add_channel(
            *,
            base: str,
            path: Path,
            unit: str,
            divisor: float,
        ) -> None:
            key = base
            suffix = 2
            while key in seen:
                key = f"{base}_{suffix}"
                suffix += 1
            seen.add(key)
            result.append(SysfsChannel(key, path, unit, divisor))

        specifications = (
            ("temp", "celsius", 1000.0),
            ("power", "watts", 1_000_000.0),
            ("energy", "joules", 1_000_000.0),
            ("fan", "rpm", 1.0),
        )
        for hwmon in sorted(Path("/sys/class/hwmon").glob("hwmon*")):
            try:
                chip = (hwmon / "name").read_text().strip()
            except OSError:
                chip = hwmon.name
            for prefix, unit, divisor in specifications:
                for path in sorted(hwmon.glob(f"{prefix}[0-9]*_input")):
                    stem = path.name.removesuffix("_input")
                    label_path = path.with_name(f"{stem}_label")
                    try:
                        label = label_path.read_text().strip()
                    except OSError:
                        label = stem
                    add_channel(
                        base=f"hwmon.{_slug(chip)}.{_slug(label)}",
                        path=path,
                        unit=unit,
                        divisor=divisor,
                    )
        powercap = Path("/sys/class/powercap")
        if powercap.is_dir():
            for path in sorted(powercap.rglob("energy_uj")):
                try:
                    zone = (path.parent / "name").read_text().strip()
                except OSError:
                    zone = path.parent.name
                add_channel(
                    base=f"powercap.{_slug(zone)}.energy",
                    path=path,
                    unit="joules",
                    divisor=1_000_000.0,
                )
            for path in sorted(powercap.rglob("power_uw")):
                try:
                    zone = (path.parent / "name").read_text().strip()
                except OSError:
                    zone = path.parent.name
                add_channel(
                    base=f"powercap.{_slug(zone)}.power",
                    path=path,
                    unit="watts",
                    divisor=1_000_000.0,
                )
        return result

    def sample(self) -> dict[str, Any]:
        values: dict[str, Any] = {}
        for channel in self.channels:
            try:
                values[channel.key] = float(channel.path.read_text().strip()) / channel.divisor
            except (OSError, ValueError):
                values[channel.key] = None
        frequencies: list[float] = []
        for path in self.frequency_paths:
            try:
                frequencies.append(float(path.read_text().strip()) / 1000.0)
            except (OSError, ValueError):
                continue
        values["cpu_frequency_average_mhz"] = (
            statistics.fmean(frequencies) if frequencies else None
        )
        return values

    def metadata(self) -> list[dict[str, str]]:
        return [
            {
                "native_name": channel.key,
                "unit": channel.unit,
                "source": str(channel.path),
            }
            for channel in self.channels
        ]


def _command_version(command: list[str]) -> str | None:
    try:
        result = subprocess.run(
            command,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            timeout=15,
            check=False,
        )
    except (OSError, subprocess.TimeoutExpired):
        return None
    text = (result.stdout or "").strip()
    return text.splitlines()[0] if text else None


def repository_state() -> dict[str, Any]:
    root = Path(
        os.environ.get(
            "PRISM_REPO_ROOT",
            str(Path(__file__).resolve().parents[2]),
        )
    ).resolve()
    try:
        commit = subprocess.check_output(
            ["git", "-C", str(root), "rev-parse", "HEAD"],
            text=True,
            stderr=subprocess.DEVNULL,
        ).strip()
        status = subprocess.check_output(
            ["git", "-C", str(root), "status", "--porcelain", "--untracked-files=normal"],
            text=True,
            stderr=subprocess.DEVNULL,
        )
        return {
            "repository": root.name,
            "git_commit": commit,
            "git_dirty": bool(status.strip()),
        }
    except (OSError, subprocess.CalledProcessError):
        return {
            "repository": root.name,
            "git_commit": None,
            "git_dirty": None,
        }


def sanitized_platform_snapshot(platform_id: str) -> dict[str, Any]:
    snapshot: dict[str, Any] = {
        "platform_id": platform_id,
        "architecture": platform.machine(),
        "operating_system": platform.system(),
        "operating_system_release": platform.release(),
        "operating_system_version": platform.version(),
        "logical_cpu_count": psutil.cpu_count(logical=True),
        "physical_cpu_count": psutil.cpu_count(logical=False),
        "memory_bytes": psutil.virtual_memory().total,
        "python_version": platform.python_version(),
        "psutil_version": psutil.__version__,
        "capture_utc": utc_now(),
        "privacy": "serial numbers, hardware UUIDs, hostname, and user identity omitted",
    }
    if platform.system() == "Darwin":
        try:
            result = subprocess.run(
                ["system_profiler", "-json", "SPHardwareDataType"],
                stdout=subprocess.PIPE,
                stderr=subprocess.DEVNULL,
                text=True,
                timeout=30,
                check=False,
            )
            payload = json.loads(result.stdout)
            hardware = payload.get("SPHardwareDataType", [{}])[0]
            snapshot["hardware"] = {
                "model_name": hardware.get("machine_name"),
                "model_identifier": hardware.get("machine_model"),
                "chip": hardware.get("chip_type"),
                "physical_memory": hardware.get("physical_memory"),
            }
        except (OSError, ValueError, subprocess.TimeoutExpired):
            snapshot["hardware"] = {}
        snapshot["tools"] = {
            "macmon": _command_version(["macmon", "--version"]),
            "xctrace": _command_version(["xcrun", "xctrace", "version"]),
        }
    elif platform.system() == "Linux":
        model = None
        try:
            for line in Path("/proc/cpuinfo").read_text().splitlines():
                if line.lower().startswith("model name"):
                    model = line.split(":", 1)[1].strip()
                    break
        except OSError:
            pass
        os_release: dict[str, str] = {}
        try:
            for line in Path("/etc/os-release").read_text().splitlines():
                if "=" in line:
                    key, value = line.split("=", 1)
                    if key in {"ID", "VERSION_ID", "PRETTY_NAME"}:
                        os_release[key.lower()] = value.strip().strip('"')
        except OSError:
            pass
        snapshot["hardware"] = {"cpu_model": model}
        snapshot["linux_distribution"] = os_release
        snapshot["tools"] = {
            "perf": _command_version(["perf", "--version"]),
            "sensors": _command_version(["sensors", "--version"]),
        }
    return safe_json(snapshot)


def flatten_leaves(value: Any, prefix: str = "") -> dict[str, Any]:
    output: dict[str, Any] = {}
    if isinstance(value, dict):
        for key, item in value.items():
            child = f"{prefix}.{key}" if prefix else str(key)
            output.update(flatten_leaves(item, child))
    elif isinstance(value, list):
        for index, item in enumerate(value):
            child = f"{prefix}.{index}" if prefix else str(index)
            output.update(flatten_leaves(item, child))
    else:
        output[prefix] = value
    return output


def sanitize_enriched_sample(
    platform_id: str,
    value: dict[str, Any] | None,
) -> tuple[dict[str, Any] | None, list[str]]:
    """Preserve raw collector output separately and null impossible derived values."""

    if value is None:
        return None, []
    sample = safe_json(value)
    flags: list[str] = []
    if platform_id == "M2_MACOS":
        temperatures = sample.get("temp")
        if isinstance(temperatures, dict):
            for name, raw in list(temperatures.items()):
                if isinstance(raw, (int, float)) and not (
                    QUALITY_CONTRACT["apple_temperature_min_celsius"]
                    <= float(raw)
                    <= QUALITY_CONTRACT["apple_temperature_max_celsius"]
                ):
                    temperatures[name] = None
                    flags.append(f"out_of_range:enriched.temp.{name}")
        for name in (
            "all_power",
            "ane_power",
            "cpu_power",
            "gpu_power",
            "gpu_ram_power",
            "ram_power",
            "sys_power",
        ):
            raw = sample.get(name)
            if isinstance(raw, (int, float)) and float(raw) < 0:
                sample[name] = None
                flags.append(f"out_of_range:enriched.{name}")
        for name in ("pcpu_usage", "ecpu_usage", "gpu_usage"):
            raw = sample.get(name)
            if (
                isinstance(raw, list)
                and len(raw) > 1
                and isinstance(raw[1], (int, float))
                and not (0.0 <= float(raw[1]) <= 1.0)
            ):
                raw[1] = None
                flags.append(f"out_of_range:enriched.{name}.1")
    return sample, flags


def channel_group(name: str) -> SemanticGroup:
    lower = name.lower()
    if any(token in lower for token in ("gpu", "ane", "accelerator")):
        return SemanticGroup.ACCELERATOR
    if any(token in lower for token in ("temperature", "temp", "power", "energy", "fan")):
        return SemanticGroup.THERMAL_POWER
    if any(token in lower for token in ("memory", "swap", "ram")):
        return SemanticGroup.MEMORY
    if any(token in lower for token in ("disk", "network", "io", "bytes_", "packets_")):
        return SemanticGroup.IO
    if "availability" in lower or "missing" in lower:
        return SemanticGroup.AVAILABILITY
    return SemanticGroup.COMPUTE


def channel_unit(name: str) -> str:
    lower = name.lower()
    if lower.endswith("timestamp") or lower.endswith("ts_utc"):
        return "ISO-8601"
    if "temperature" in lower or "_temp" in lower or ".temp" in lower:
        return "celsius"
    if "power" in lower:
        return "watts"
    if "energy" in lower:
        return "joules"
    if "frequency" in lower or lower.endswith("_mhz") or lower.endswith("usage.0"):
        return "megahertz"
    if "percent" in lower:
        return "percent"
    if lower.endswith("usage.1"):
        return "fraction"
    if "bytes_per_second" in lower:
        return "bytes/second"
    if ("memory." in lower or "swap." in lower) and not lower.endswith(".percent"):
        return "bytes"
    if "bytes" in lower or "ram_" in lower:
        return "bytes"
    if "seconds" in lower or lower.endswith("_s"):
        return "seconds"
    if "fan" in lower:
        return "rpm"
    return "count_or_native"


def channel_registry(samples: Iterable[dict[str, Any]], period: float) -> list[dict[str, Any]]:
    names: set[str] = set()
    for sample in samples:
        for branch in ("host", "enriched"):
            value = sample.get(branch)
            if value is not None:
                names.update(f"{branch}.{name}" for name in flatten_leaves(value))
    registry = []
    for name in sorted(names):
        registry.append(
            {
                "native_name": name,
                "unit": channel_unit(name),
                "semantic_group": channel_group(name).value,
                "source": "psutil" if name.startswith("host.") else "platform_native",
                "sampling_period_seconds": period,
                "missingness": "null means unavailable or unreadable; never imputed as zero",
            }
        )
    return registry


def write_checksums(run_dir: Path) -> None:
    excluded = {"checksums.sha256", "validation.json"}
    lines: list[str] = []
    for path in sorted(item for item in run_dir.rglob("*") if item.is_file()):
        if path.name in excluded:
            continue
        lines.append(f"{file_sha256(path)}  {path.relative_to(run_dir).as_posix()}")
    (run_dir / "checksums.sha256").write_text("\n".join(lines) + "\n", encoding="utf-8")


def file_sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def _nested_value(value: Any, dotted_name: str) -> Any:
    current = value
    for part in dotted_name.split("."):
        if isinstance(current, dict):
            current = current.get(part)
        elif isinstance(current, list) and part.isdigit():
            index = int(part)
            current = current[index] if index < len(current) else None
        else:
            return None
    return current


def telemetry_quality_summary(
    samples: list[dict[str, Any]],
    platform_id: str,
    profile: str = "enriched",
) -> dict[str, Any]:
    usable_samples = [
        sample
        for sample in samples
        if not sample.get("availability", {}).get("interruption_injected", False)
    ]
    critical = (
        (
            "enriched.cpu_power",
            "enriched.sys_power",
            "enriched.pcpu_usage.1",
            "enriched.temp.cpu_temp_avg",
        )
        if platform_id == "M2_MACOS" and profile == "enriched"
        else (
            "host.cpu_percent",
            "host.memory.percent",
            "host.load_average.0",
        )
    )
    channels: dict[str, Any] = {}
    denominator = len(usable_samples)
    for name in critical:
        values = [_nested_value(sample, name) for sample in usable_samples]
        if name == "enriched.temp.cpu_temp_avg":
            valid = [
                isinstance(value, (int, float))
                and QUALITY_CONTRACT["apple_temperature_min_celsius"]
                <= float(value)
                <= QUALITY_CONTRACT["apple_temperature_max_celsius"]
                for value in values
            ]
        elif name in {"enriched.cpu_power", "enriched.sys_power"}:
            valid = [
                isinstance(value, (int, float)) and float(value) >= 0
                for value in values
            ]
        elif name == "enriched.pcpu_usage.1":
            valid = [
                isinstance(value, (int, float)) and 0.0 <= float(value) <= 1.0
                for value in values
            ]
        else:
            valid = [value is not None for value in values]
        count = sum(valid)
        channels[name] = {
            "non_null_samples": count,
            "eligible_samples": denominator,
            "non_null_fraction": count / denominator if denominator else 0.0,
        }
    flag_counts: dict[str, int] = {}
    for sample in samples:
        for flag in sample.get("availability", {}).get("quality_flags", []):
            flag_counts[str(flag)] = flag_counts.get(str(flag), 0) + 1
    if platform_id == "M2_MACOS":
        recorded_temperature_flags = flag_counts.get(
            "out_of_range:enriched.temp.cpu_temp_avg",
            0,
        )
        observed_temperature_flags = sum(
            isinstance(value, (int, float))
            and not (
                QUALITY_CONTRACT["apple_temperature_min_celsius"]
                <= float(value)
                <= QUALITY_CONTRACT["apple_temperature_max_celsius"]
            )
            for value in (
                _nested_value(sample, "enriched.temp.cpu_temp_avg")
                for sample in usable_samples
            )
        )
        if observed_temperature_flags > recorded_temperature_flags:
            flag_counts["out_of_range:enriched.temp.cpu_temp_avg"] = (
                observed_temperature_flags
            )
    return {
        "critical_channels": channels,
        "quality_flag_counts": flag_counts,
        "interruption_masked_samples": len(samples) - len(usable_samples),
    }


def verify_checksums(run_dir: Path) -> list[str]:
    failures: list[str] = []
    manifest = run_dir / "checksums.sha256"
    if not manifest.is_file():
        return ["checksums.sha256 is missing"]
    for line in manifest.read_text().splitlines():
        expected, relative = line.split("  ", 1)
        path = run_dir / relative
        if not path.is_file():
            failures.append(f"missing checksum target: {relative}")
            continue
        actual = file_sha256(path)
        if actual != expected:
            failures.append(f"checksum mismatch: {relative}")
    return failures


def validate_run(run_dir: Path, verify_hashes: bool = True) -> dict[str, Any]:
    failures: list[str] = []
    warnings: list[str] = []
    required = (
        "platform.json",
        "collection.json",
        "telemetry.jsonl",
        "events.jsonl",
        "channels.json",
    )
    for filename in required:
        if not (run_dir / filename).is_file():
            failures.append(f"missing required file: {filename}")
    if failures:
        return {"valid": False, "failures": failures, "validated_utc": utc_now()}

    collection = json.loads((run_dir / "collection.json").read_text())
    samples = [
        json.loads(line)
        for line in (run_dir / "telemetry.jsonl").read_text().splitlines()
        if line.strip()
    ]
    events = [
        json.loads(line)
        for line in (run_dir / "events.jsonl").read_text().splitlines()
        if line.strip()
    ]
    expected = int(collection["expected_samples"])
    if len(samples) != expected:
        failures.append(f"sample count {len(samples)} does not equal expected {expected}")
    relative = [float(sample["t_rel_seconds"]) for sample in samples]
    if any(right <= left for left, right in zip(relative, relative[1:])):
        failures.append("sample relative timestamps are not strictly increasing")
    if len(relative) > 2:
        deltas = [right - left for left, right in zip(relative, relative[1:])]
        median = statistics.median(deltas)
        target = 1.0 / float(collection["sampling_hz"])
        if not (target * 0.5 <= median <= target * 1.5):
            failures.append(
                f"median sampling interval {median:.4f}s is outside tolerance around {target:.4f}s"
            )
        if max(deltas) > target * 3:
            failures.append(
                f"maximum sampling gap {max(deltas):.4f}s exceeds three periods"
            )
        expected_span = (expected - 1) * target
        observed_span = relative[-1] - relative[0]
        span_tolerance = max(0.5, expected_span * 0.05)
        if abs(observed_span - expected_span) > span_tolerance:
            failures.append(
                f"observed sample span {observed_span:.4f}s differs from expected "
                f"{expected_span:.4f}s by more than {span_tolerance:.4f}s"
            )
    event_names = {event.get("event") for event in events}
    if "workload_started" not in event_names:
        failures.append("workload_started event is missing")
    scenario = collection["scenario"]
    if scenario == "CONTROLLED_CRASH":
        if "controlled_crash_injected" not in event_names:
            failures.append("controlled_crash_injected event is missing")
    elif scenario == "TELEMETRY_INTERRUPTION":
        for required_event in (
            "telemetry_interruption_started",
            "telemetry_interruption_ended",
        ):
            if required_event not in event_names:
                failures.append(f"{required_event} event is missing")
        interrupted = [
            sample
            for sample in samples
            if sample.get("availability", {}).get("interruption_injected", False)
        ]
        if not interrupted:
            failures.append("telemetry interruption produced no masked samples")
        if interrupted and len(interrupted) == len(samples):
            failures.append("telemetry interruption never recovered before run end")
    elif scenario != "NOMINAL" and "stressor_started" not in event_names:
        failures.append("stressor_started event is missing")
    if collection.get("status") != "complete":
        failures.append(f"collection status is {collection.get('status')!r}, not 'complete'")
    if verify_hashes:
        failures.extend(verify_checksums(run_dir))
    null_enriched = sum(sample.get("enriched") is None for sample in samples)
    platform_id = str(collection.get("platform_id", ""))
    quality = (
        telemetry_quality_summary(
            samples,
            platform_id,
            str(collection.get("profile", "enriched")),
        )
        if platform_id in {"M2_MACOS", "EPYC_LINUX"}
        else {
            "critical_channels": {},
            "quality_flag_counts": {},
            "interruption_masked_samples": 0,
        }
    )
    for name, metrics in quality["critical_channels"].items():
        fraction = float(metrics["non_null_fraction"])
        if name == "enriched.temp.cpu_temp_avg":
            minimum_fraction = QUALITY_CONTRACT[
                "apple_temperature_minimum_fraction"
            ]
        else:
            minimum_fraction = (
                QUALITY_CONTRACT["primary_channel_minimum_fraction"]
                if collection.get("purpose") == "production"
                else 0.70
            )
        if fraction < minimum_fraction:
            failures.append(
                f"critical channel {name} non-null fraction {fraction:.3f} "
                f"is below {minimum_fraction:.2f}"
            )
        elif fraction < 0.95:
            warnings.append(
                f"critical channel {name} non-null fraction is {fraction:.3f}"
            )
    flagged = sum(quality["quality_flag_counts"].values())
    flagged_fraction = flagged / len(samples) if samples else 0.0
    if flagged_fraction > QUALITY_CONTRACT["maximum_quality_flag_fraction"]:
        failures.append(
            f"quality flags affect {flagged_fraction:.3f} of samples, above "
            f"{QUALITY_CONTRACT['maximum_quality_flag_fraction']:.2f}"
        )
    elif flagged:
        warnings.append(
            f"detected {flagged} impossible enriched values; they are excluded "
            "from critical-channel quality coverage"
        )
    return {
        "valid": not failures,
        "failures": failures,
        "warnings": warnings,
        "validated_utc": utc_now(),
        "sample_count": len(samples),
        "expected_samples": expected,
        "duration_observed_seconds": relative[-1] - relative[0] if len(relative) > 1 else 0,
        "enriched_missing_samples": null_enriched,
        "quality": quality,
        "checksums_verified": verify_hashes,
    }


def _event(stream: Any, start: float, name: str, **details: Any) -> None:
    append_jsonl(
        stream,
        {
            "event": name,
            "ts_utc": utc_now(),
            "t_rel_seconds": time.monotonic() - start,
            **details,
        },
    )


def _start_harness(role: str, name: str, log_path: Path) -> tuple[subprocess.Popen[str], Any]:
    log_handle = log_path.open("w", encoding="utf-8")
    environment = os.environ.copy()
    threads = environment.get("PRISM_NUM_THREADS", "1")
    environment.update(
        {
            "OMP_NUM_THREADS": threads,
            "OPENBLAS_NUM_THREADS": threads,
            "MKL_NUM_THREADS": threads,
            "VECLIB_MAXIMUM_THREADS": threads,
            "NUMEXPR_NUM_THREADS": threads,
        }
    )
    process = subprocess.Popen(
        [
            sys.executable,
            "-m",
            "prism_slm.workload_harness",
            "--role",
            role,
            "--name",
            name,
        ],
        stdout=log_handle,
        stderr=subprocess.STDOUT,
        text=True,
        env=environment,
    )
    return process, log_handle


def _stop_process(process: subprocess.Popen[str] | None, log_handle: Any = None) -> int | None:
    code = None
    if process is not None:
        if process.poll() is None:
            process.terminate()
            try:
                process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait(timeout=10)
        code = process.returncode
    if log_handle is not None:
        log_handle.close()
    return code


@dataclass(frozen=True)
class CollectionRequest:
    platform_id: str
    workload: str
    scenario: str
    repetition: int
    split: str
    duration_seconds: int
    sampling_hz: float
    warmup_seconds: float
    interruption_seconds: float
    purpose: str
    profile: str
    output_root: Path
    run_id: str | None = None
    macmon_executable: str = "macmon"
    enable_xctrace: bool = False
    xctrace_template: str = "Time Profiler"


def collect(request: CollectionRequest) -> Path:
    workload = request.workload.upper()
    scenario = request.scenario.upper()
    current_system = platform.system()
    expected_systems = {"M2_MACOS": "Darwin", "EPYC_LINUX": "Linux"}
    if request.platform_id not in expected_systems:
        raise ValueError(f"unsupported platform_id: {request.platform_id}")
    if current_system != expected_systems[request.platform_id]:
        raise ValueError(
            f"platform_id {request.platform_id} requires {expected_systems[request.platform_id]}, "
            f"but the current system is {current_system}"
        )
    if request.profile not in {"portable", "enriched"}:
        raise ValueError(f"unsupported collection profile: {request.profile}")
    if request.purpose not in {"smoke", "production"}:
        raise ValueError(f"unsupported collection purpose: {request.purpose}")
    if workload not in WORKLOADS:
        raise ValueError(f"unsupported workload: {workload}")
    if scenario not in SCENARIOS:
        raise ValueError(f"unsupported executable scenario: {scenario}")
    if request.duration_seconds < 5:
        raise ValueError("duration_seconds must be at least 5")
    if request.sampling_hz <= 0 or request.sampling_hz > 20:
        raise ValueError("sampling_hz must be in (0, 20]")
    if scenario != "NOMINAL" and not (0 <= request.warmup_seconds < request.duration_seconds):
        raise ValueError("warmup_seconds must be within the run for anomalous scenarios")
    if scenario == "TELEMETRY_INTERRUPTION" and not (
        0 < request.interruption_seconds
        < request.duration_seconds - request.warmup_seconds
    ):
        raise ValueError(
            "interruption_seconds must be positive and leave time for recovery"
        )
    if scenario == "TELEMETRY_INTERRUPTION" and request.profile != "enriched":
        raise ValueError("TELEMETRY_INTERRUPTION requires the enriched profile")
    software_state = repository_state()
    if request.purpose == "production" and software_state.get("git_dirty") is not False:
        raise RuntimeError(
            "production collection requires a clean, committed Git checkout"
        )

    timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    run_id = request.run_id or (
        f"{request.purpose}__{request.platform_id.lower()}__{workload.lower()}__"
        f"{scenario.lower()}__r{request.repetition:02d}__{timestamp}"
    )
    run_dir = request.output_root / request.platform_id / utc_date() / run_id
    if run_dir.exists():
        raise FileExistsError(f"run directory already exists: {run_dir}")
    run_dir.mkdir(parents=True)

    period = 1.0 / request.sampling_hz
    expected_samples = int(round(request.duration_seconds * request.sampling_hz))
    collector_process = psutil.Process(os.getpid())
    collector_cpu_start = collector_process.cpu_times()
    collection_record: dict[str, Any] = {
        "run_id": run_id,
        "platform_id": request.platform_id,
        "workload": workload,
        "scenario": scenario,
        "repetition": request.repetition,
        "planned_split": request.split,
        "purpose": request.purpose,
        "profile": request.profile,
        "duration_seconds": request.duration_seconds,
        "sampling_hz": request.sampling_hz,
        "warmup_seconds": request.warmup_seconds,
        "interruption_seconds": request.interruption_seconds,
        "expected_samples": expected_samples,
        "start_utc": utc_now(),
        "status": "collecting",
        "command": sys.argv,
        "software": software_state,
        "workload_protocol": {
            "num_threads": int(os.environ.get("PRISM_NUM_THREADS", "1")),
            "stress_memory_mb": int(os.environ.get("PRISM_STRESS_MB", "128")),
        },
        "quality_contract": QUALITY_CONTRACT,
    }
    dump_json(run_dir / "platform.json", sanitized_platform_snapshot(request.platform_id))
    dump_json(run_dir / "collection.json", collection_record)

    host = HostSampler()
    macmon: MacmonStream | None = None
    linux_sysfs: LinuxSysfsSampler | None = None
    workload_process: subprocess.Popen[str] | None = None
    workload_log: Any = None
    stressor_process: subprocess.Popen[str] | None = None
    stressor_log: Any = None
    trace_process: subprocess.Popen[str] | None = None
    trace_log: Any = None
    samples_for_registry: list[dict[str, Any]] = []
    setup_start = time.monotonic()
    start = setup_start
    collection_error: str | None = None
    controlled_crash_injected = False
    telemetry_interruption_started = False
    telemetry_interruption_ended = False

    with (run_dir / "events.jsonl").open("w", encoding="utf-8") as events, (
        run_dir / "telemetry.jsonl"
    ).open("w", encoding="utf-8") as telemetry:
        try:
            enriched_event: dict[str, Any] | None = None
            if platform.system() == "Darwin" and request.profile == "enriched":
                executable = shutil.which(request.macmon_executable)
                if executable is None:
                    raise RuntimeError(f"macmon not found: {request.macmon_executable}")
                macmon = MacmonStream(
                    executable,
                    int(round(period * 1000)),
                    run_dir / "macmon.log",
                    run_dir / "macmon-raw.jsonl",
                )
                macmon.start()
                enriched_event = {"source": "macmon"}
            elif platform.system() == "Linux" and request.profile == "enriched":
                linux_sysfs = LinuxSysfsSampler()
                enriched_event = {
                    "source": "linux_sysfs",
                    "discovered_channels": len(linux_sysfs.channels),
                }

            # The research clock begins only after platform collectors are
            # ready. Setup latency must not compress early sample intervals or
            # shift the declared stressor onset.
            start = time.monotonic()
            collection_record["start_utc"] = utc_now()
            _event(events, start, "collector_started", profile=request.profile)
            if enriched_event is not None:
                _event(events, start, "enriched_collector_started", **enriched_event)
            if request.enable_xctrace:
                if platform.system() != "Darwin":
                    raise RuntimeError("xctrace is supported only on macOS")
                trace_log = (run_dir / "xctrace.log").open("w", encoding="utf-8")
                trace_process = subprocess.Popen(
                    [
                        "xcrun",
                        "xctrace",
                        "record",
                        "--template",
                        request.xctrace_template,
                        "--all-processes",
                        "--time-limit",
                        f"{request.duration_seconds}s",
                        "--output",
                        str(run_dir / "xctrace.trace"),
                        "--no-prompt",
                    ],
                    stdout=trace_log,
                    stderr=subprocess.STDOUT,
                    text=True,
                )
                _event(
                    events,
                    start,
                    "trace_collector_started",
                    source="xctrace",
                    template=request.xctrace_template,
                )

            workload_process, workload_log = _start_harness(
                "workload", workload, run_dir / "workload.log"
            )
            _event(events, start, "workload_started", workload=workload)

            next_sample = start
            for sequence in range(expected_samples):
                now = time.monotonic()
                elapsed = now - start
                if (
                    scenario == "CONTROLLED_CRASH"
                    and not controlled_crash_injected
                    and elapsed >= request.warmup_seconds
                ):
                    assert workload_process is not None
                    workload_process.kill()
                    workload_process.wait(timeout=10)
                    controlled_crash_injected = True
                    _event(
                        events,
                        start,
                        "controlled_crash_injected",
                        target="workload_child",
                        signal="SIGKILL",
                        exit_code=workload_process.returncode,
                    )
                elif (
                    scenario == "TELEMETRY_INTERRUPTION"
                    and not telemetry_interruption_started
                    and elapsed >= request.warmup_seconds
                ):
                    telemetry_interruption_started = True
                    _event(
                        events,
                        start,
                        "telemetry_interruption_started",
                        source=(
                            "macmon"
                            if macmon is not None
                            else "linux_sysfs"
                            if linux_sysfs is not None
                            else "enriched"
                        ),
                        planned_duration_seconds=request.interruption_seconds,
                    )
                elif (
                    scenario == "TELEMETRY_INTERRUPTION"
                    and telemetry_interruption_started
                    and not telemetry_interruption_ended
                    and elapsed
                    >= request.warmup_seconds + request.interruption_seconds
                ):
                    telemetry_interruption_ended = True
                    _event(
                        events,
                        start,
                        "telemetry_interruption_ended",
                        planned_duration_seconds=request.interruption_seconds,
                    )
                elif (
                    scenario in STRESSORS
                    and scenario != "NOMINAL"
                    and stressor_process is None
                ):
                    if elapsed >= request.warmup_seconds:
                        stressor_process, stressor_log = _start_harness(
                            "stressor", scenario, run_dir / "stressor.log"
                        )
                        _event(events, start, "stressor_started", scenario=scenario)

                if workload_process.poll() is not None and not (
                    scenario == "CONTROLLED_CRASH" and controlled_crash_injected
                ):
                    raise RuntimeError(
                        f"workload exited before collection ended with code {workload_process.returncode}"
                    )
                if stressor_process is not None and stressor_process.poll() is not None:
                    raise RuntimeError(
                        f"stressor exited before collection ended with code {stressor_process.returncode}"
                    )

                enriched_raw: dict[str, Any] | None = None
                source = None
                if macmon is not None:
                    enriched_raw = macmon.latest()
                    source = "macmon"
                elif linux_sysfs is not None:
                    enriched_raw = linux_sysfs.sample()
                    source = "linux_sysfs"
                enriched, quality_flags = sanitize_enriched_sample(
                    request.platform_id,
                    enriched_raw,
                )
                interruption_active = (
                    scenario == "TELEMETRY_INTERRUPTION"
                    and telemetry_interruption_started
                    and not telemetry_interruption_ended
                )
                if interruption_active:
                    enriched = None

                sample = {
                    "sequence": sequence,
                    "ts_utc": utc_now(),
                    "t_rel_seconds": time.monotonic() - start,
                    "host": host.sample(),
                    "enriched": enriched,
                    "availability": {
                        "host_available": True,
                        "enriched_available": enriched is not None,
                        "enriched_raw_available": enriched_raw is not None,
                        "enriched_source": source,
                        "interruption_injected": interruption_active,
                        "quality_flags": quality_flags,
                    },
                }
                append_jsonl(telemetry, sample)
                if len(samples_for_registry) < 10:
                    samples_for_registry.append(sample)

                next_sample += period
                time.sleep(max(0.0, next_sample - time.monotonic()))

            _event(events, start, "sampling_completed", samples=expected_samples)
        except BaseException as exc:
            collection_error = f"{type(exc).__name__}: {exc}"
            (run_dir / "collector_error.log").write_text(
                traceback.format_exc(), encoding="utf-8"
            )
            _event(events, start, "collection_failed", error=collection_error)
        finally:
            stress_code = _stop_process(stressor_process, stressor_log)
            workload_code = _stop_process(workload_process, workload_log)
            if macmon is not None:
                macmon.stop()
            trace_code = None
            if trace_process is not None:
                try:
                    trace_code = trace_process.wait(timeout=30)
                except subprocess.TimeoutExpired:
                    trace_process.terminate()
                    trace_code = trace_process.wait(timeout=10)
                if trace_code != 0 and collection_error is None:
                    collection_error = f"xctrace exited with code {trace_code}"
            if trace_log is not None:
                trace_log.close()
            _event(
                events,
                start,
                "collectors_stopped",
                workload_exit_code=workload_code,
                stressor_exit_code=stress_code,
                trace_exit_code=trace_code,
            )

    elapsed_seconds = time.monotonic() - start
    collector_cpu_end = collector_process.cpu_times()
    telemetry_bytes = (run_dir / "telemetry.jsonl").stat().st_size
    events_bytes = (run_dir / "events.jsonl").stat().st_size
    native_bytes = sum(
        path.stat().st_size
        for path in run_dir.glob("*-raw.jsonl")
        if path.is_file()
    )
    collection_record.update(
        {
            "end_utc": utc_now(),
            "elapsed_seconds": elapsed_seconds,
            "setup_elapsed_seconds": start - setup_start,
            "status": "failed" if collection_error else "complete",
            "error": collection_error,
            "xctrace_enabled": request.enable_xctrace,
            "xctrace_template": request.xctrace_template if request.enable_xctrace else None,
            "collection_cost": {
                "collector_cpu_seconds": (
                    collector_cpu_end.user
                    + collector_cpu_end.system
                    - collector_cpu_start.user
                    - collector_cpu_start.system
                ),
                "telemetry_bytes": telemetry_bytes,
                "events_bytes": events_bytes,
                "native_raw_bytes": native_bytes,
                "telemetry_bytes_per_second": (
                    telemetry_bytes / elapsed_seconds if elapsed_seconds > 0 else None
                ),
            },
        }
    )
    dump_json(run_dir / "collection.json", collection_record)
    dump_json(
        run_dir / "channels.json",
        {
            "channels": channel_registry(samples_for_registry, period),
            "linux_sysfs_channels": linux_sysfs.metadata() if linux_sysfs else [],
        },
    )
    validation = validate_run(run_dir, verify_hashes=False)
    dump_json(run_dir / "validation.json", validation)
    write_checksums(run_dir)
    validation = validate_run(run_dir, verify_hashes=True)
    dump_json(run_dir / "validation.json", validation)

    if collection_error:
        raise RuntimeError(f"collection failed; preserved run at {run_dir}: {collection_error}")
    if not validation["valid"]:
        raise RuntimeError(f"collection validation failed at {run_dir}: {validation['failures']}")
    return run_dir


def update_collection_plan(plan_path: Path, run_dir: Path) -> None:
    collection = json.loads((run_dir / "collection.json").read_text())
    validation = json.loads((run_dir / "validation.json").read_text())
    rows: list[dict[str, str]]
    with plan_path.open(newline="", encoding="utf-8") as stream:
        reader = csv.DictReader(stream)
        fieldnames = list(reader.fieldnames or [])
        rows = list(reader)
    matched = False
    for row in rows:
        if row["run_id"] == collection["run_id"]:
            row["status"] = "valid" if validation["valid"] else "replacement_required"
            row["start_utc"] = collection["start_utc"]
            row["end_utc"] = collection["end_utc"]
            row["valid_rows"] = str(validation["sample_count"])
            if collection["scenario"] == "NOMINAL" and validation["valid"]:
                row["benign_hours"] = (
                    f"{float(validation['duration_observed_seconds']) / 3600:.6f}"
                )
            row["exclusion_reason"] = "; ".join(validation["failures"])
            warnings = validation.get("warnings", [])
            if warnings:
                prefix = f"{row['notes']}; " if row.get("notes") else ""
                row["notes"] = prefix + "validation warnings: " + "; ".join(warnings)
            matched = True
            break
    if not matched:
        raise ValueError(f"run_id {collection['run_id']} is not present in {plan_path}")
    with plan_path.open("w", newline="", encoding="utf-8") as stream:
        writer = csv.DictWriter(stream, fieldnames=fieldnames, lineterminator="\n")
        writer.writeheader()
        writer.writerows(rows)


### `scripts/preflight.py`


In [ ]:
%%writefile .prism_runtime/scripts/preflight.py
#!/usr/bin/env python3
"""Validate the PRISM experiment contract before data collection."""

from __future__ import annotations

import argparse
from datetime import date
from pathlib import Path
import tomllib


REQUIRED_TOP_LEVEL = {
    "study",
    "platforms",
    "matrix",
    "collection",
    "splits",
    "gates",
    "long_benign",
}

EXTENSION_REQUIRED_SCENARIOS = {
    "NOMINAL",
    "ATOMIC",
    "BRANCH",
    "CACHE",
    "MEMBW",
    "TLB",
    "CONTROLLED_CRASH",
    "TELEMETRY_INTERRUPTION",
}
EXTENSION_TARGETED_SCENARIOS = {
    "THERMAL_SHIFT",
    "POWER_SHIFT",
    "DEGRADATION_PROXY",
}


def validate(config: dict) -> list[str]:
    errors: list[str] = []
    missing = sorted(REQUIRED_TOP_LEVEL - set(config))
    if missing:
        errors.append(f"missing top-level keys: {', '.join(missing)}")

    required_platforms = [
        platform for platform in config.get("platforms", []) if platform.get("required")
    ]
    if len(required_platforms) < 2:
        errors.append("at least two platforms must be required")

    required_ids = [platform.get("id") for platform in required_platforms]
    if len(required_ids) != len(set(required_ids)):
        errors.append("required platform IDs must be unique")

    repetitions = config.get("collection", {}).get(
        "minimum_independent_repetitions", 0
    )
    if repetitions < 3:
        errors.append("minimum independent repetitions must be at least 3")

    matrix = config.get("matrix", {})
    required_scenarios = set(matrix.get("required_scenarios", []))
    missing_required = sorted(EXTENSION_REQUIRED_SCENARIOS - required_scenarios)
    if missing_required:
        errors.append(
            "required scenarios do not cover the extension: "
            + ", ".join(missing_required)
        )
    targeted_scenarios = set(matrix.get("targeted_scenarios", []))
    missing_targeted = sorted(EXTENSION_TARGETED_SCENARIOS - targeted_scenarios)
    if missing_targeted:
        errors.append(
            "targeted scenarios do not cover the extension: "
            + ", ".join(missing_targeted)
        )
    if len(matrix.get("targeted_scenario_workloads", [])) < 2:
        errors.append("targeted extension scenarios require at least two workloads")

    collection = config.get("collection", {})
    if collection.get("target_run_duration_seconds", 0) < 720:
        errors.append("production matrix runs must be at least 720 seconds")
    if collection.get("target_benign_hours_per_required_platform", 0) < 12:
        errors.append("benign target must be at least 12 hours per platform")

    long_benign = config.get("long_benign", {})
    if not long_benign.get("required"):
        errors.append("long-benign collection must be required")
    long_workloads = long_benign.get("workloads", [])
    sessions = long_benign.get("sessions_per_workload", 0)
    session_seconds = long_benign.get("session_duration_seconds", 0)
    matrix_benign_hours = (
        len(matrix.get("workloads", []))
        * repetitions
        * collection.get("target_run_duration_seconds", 0)
        / 3600
    )
    supplement_hours = len(long_workloads) * sessions * session_seconds / 3600
    if matrix_benign_hours + supplement_hours < collection.get(
        "target_benign_hours_per_required_platform", 0
    ):
        errors.append(
            "matrix nominal runs plus long-benign sessions do not meet the "
            "per-platform benign-hour target"
        )

    if config.get("splits", {}).get("unit") != "run_id":
        errors.append("split unit must be run_id")
    if not config.get("splits", {}).get("forbid_window_level_split"):
        errors.append("window-level splitting must be forbidden")

    fractions = [
        config.get("splits", {}).get("calibration_fraction", 0),
        config.get("splits", {}).get("development_fraction", 0),
        config.get("splits", {}).get("locked_test_fraction", 0),
    ]
    if abs(sum(fractions) - 1.0) > 1e-9:
        errors.append("split fractions must sum to 1.0")

    gate_dates: list[date] = []
    for gate_name, raw_date in config.get("gates", {}).items():
        try:
            gate_dates.append(date.fromisoformat(str(raw_date)))
        except ValueError:
            errors.append(f"gate {gate_name} has invalid ISO date: {raw_date}")
    if gate_dates != sorted(gate_dates):
        errors.append("gate dates must be chronological")

    return errors


def main() -> int:
    parser = argparse.ArgumentParser()
    parser.add_argument("config", type=Path)
    args = parser.parse_args()

    with args.config.open("rb") as config_file:
        config = tomllib.load(config_file)
    errors = validate(config)
    if errors:
        for error in errors:
            print(f"ERROR: {error}")
        return 1

    required_platforms = [
        platform["id"] for platform in config["platforms"] if platform["required"]
    ]
    print("PRISM experiment contract is valid")
    print(f"required platforms: {', '.join(required_platforms)}")
    print(
        "minimum repetitions: "
        f"{config['collection']['minimum_independent_repetitions']}"
    )
    print(
        "extension scenarios: "
        f"{len(config['matrix']['required_scenarios'])} required + "
        f"{len(config['matrix']['targeted_scenarios'])} targeted"
    )
    matrix_benign_hours = (
        len(config["matrix"]["workloads"])
        * config["collection"]["minimum_independent_repetitions"]
        * config["collection"]["target_run_duration_seconds"]
        / 3600
    )
    long_benign = config["long_benign"]
    supplement_hours = (
        len(long_benign["workloads"])
        * long_benign["sessions_per_workload"]
        * long_benign["session_duration_seconds"]
        / 3600
    )
    print(
        "benign coverage per platform: "
        f"{matrix_benign_hours:.1f} h matrix + "
        f"{supplement_hours:.1f} h supplement = "
        f"{matrix_benign_hours + supplement_hours:.1f} h"
    )
    print(f"submission gate: {config['gates']['submission']}")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


### `scripts/make_collection_plan.py`


In [ ]:
%%writefile .prism_runtime/scripts/make_collection_plan.py
#!/usr/bin/env python3
"""Generate the predeclared PRISM run-level collection tracker."""

from __future__ import annotations

import argparse
import csv
import tomllib
from pathlib import Path


FIELDS = (
    "run_id",
    "platform_id",
    "workload",
    "scenario",
    "run_kind",
    "repetition",
    "planned_split",
    "purpose",
    "profile",
    "duration_seconds",
    "sampling_hz",
    "warmup_seconds",
    "interruption_seconds",
    "status",
    "replacement_for",
    "start_utc",
    "end_utc",
    "valid_rows",
    "benign_hours",
    "exclusion_reason",
    "notes",
)


def split_for(scenario: str, repetition: int) -> str:
    if repetition >= 3:
        return "locked_test"
    if scenario == "NOMINAL" and repetition == 1:
        return "calibration"
    return "development"


def _matrix_row(
    *,
    platform: str,
    workload: str,
    scenario: str,
    repetition: int,
    collection: dict,
    run_kind: str,
) -> dict[str, object]:
    run_id = (
        f"{platform.lower()}__{workload.lower()}__"
        f"{scenario.lower()}__r{repetition:02d}"
    )
    return {
        "run_id": run_id,
        "platform_id": platform,
        "workload": workload,
        "scenario": scenario,
        "run_kind": run_kind,
        "repetition": repetition,
        "planned_split": split_for(scenario, repetition),
        "purpose": "production",
        "profile": "enriched",
        "duration_seconds": collection["target_run_duration_seconds"],
        "sampling_hz": collection["sampling_hz"],
        "warmup_seconds": (
            0
            if scenario == "NOMINAL"
            else collection["anomaly_warmup_seconds"]
        ),
        "interruption_seconds": (
            collection["telemetry_interruption_seconds"]
            if scenario == "TELEMETRY_INTERRUPTION"
            else 0
        ),
        "status": "planned",
        "replacement_for": "",
        "start_utc": "",
        "end_utc": "",
        "valid_rows": "",
        "benign_hours": "",
        "exclusion_reason": "",
        "notes": "",
    }


def build_rows(config: dict, repetitions_override: int | None = None) -> list[dict[str, object]]:
    platforms = [
        item["id"] for item in config["platforms"] if item.get("required", False)
    ]
    matrix = config["matrix"]
    collection = config["collection"]
    workloads = matrix["workloads"]
    repetitions = repetitions_override or collection["minimum_independent_repetitions"]
    if repetitions < collection["minimum_independent_repetitions"]:
        raise ValueError("Repetitions cannot be below the declared minimum")

    rows: list[dict[str, object]] = []
    for platform in platforms:
        for workload in workloads:
            for scenario in matrix["required_scenarios"]:
                for repetition in range(1, repetitions + 1):
                    rows.append(
                        _matrix_row(
                            platform=platform,
                            workload=workload,
                            scenario=scenario,
                            repetition=repetition,
                            collection=collection,
                            run_kind="required_matrix",
                        )
                    )

        for workload in matrix["targeted_scenario_workloads"]:
            for scenario in matrix["targeted_scenarios"]:
                for repetition in range(1, repetitions + 1):
                    rows.append(
                        _matrix_row(
                            platform=platform,
                            workload=workload,
                            scenario=scenario,
                            repetition=repetition,
                            collection=collection,
                            run_kind="targeted_extension",
                        )
                    )

        long_benign = config["long_benign"]
        for workload in long_benign["workloads"]:
            for session in range(1, long_benign["sessions_per_workload"] + 1):
                rows.append(
                    {
                        "run_id": (
                            f"{platform.lower()}__{workload.lower()}__"
                            f"long_benign__s{session:02d}"
                        ),
                        "platform_id": platform,
                        "workload": workload,
                        "scenario": "NOMINAL",
                        "run_kind": "long_benign",
                        "repetition": session,
                        "planned_split": (
                            "development" if session == 1 else "locked_test"
                        ),
                        "purpose": "production",
                        "profile": "enriched",
                        "duration_seconds": long_benign["session_duration_seconds"],
                        "sampling_hz": long_benign["sampling_hz"],
                        "warmup_seconds": 0,
                        "interruption_seconds": 0,
                        "status": "planned",
                        "replacement_for": "",
                        "start_utc": "",
                        "end_utc": "",
                        "valid_rows": "",
                        "benign_hours": "",
                        "exclusion_reason": "",
                        "notes": "long-benign supplement toward 12 h/platform",
                    }
                )
    return rows


def main() -> int:
    root = Path(__file__).resolve().parents[1]
    parser = argparse.ArgumentParser()
    parser.add_argument(
        "--config",
        type=Path,
        default=root / "configs/experiment-matrix.toml",
    )
    parser.add_argument(
        "--output",
        type=Path,
        default=root / "data/collection-plan.csv",
    )
    parser.add_argument(
        "--repetitions",
        type=int,
        help="Override minimum repetitions from the experiment configuration",
    )
    parser.add_argument(
        "--overwrite",
        action="store_true",
        help="Replace an existing plan; never use after collection has started",
    )
    args = parser.parse_args()

    with args.config.open("rb") as stream:
        config = tomllib.load(stream)

    if args.output.exists() and not args.overwrite:
        raise SystemExit(
            f"Plan already exists: {args.output}. Pass --overwrite only before "
            "production collection begins."
        )
    try:
        rows = build_rows(config, args.repetitions)
    except ValueError as exc:
        raise SystemExit(str(exc)) from exc

    args.output.parent.mkdir(parents=True, exist_ok=True)
    with args.output.open("w", newline="", encoding="utf-8") as stream:
        writer = csv.DictWriter(stream, fieldnames=FIELDS, lineterminator="\n")
        writer.writeheader()
        writer.writerows(rows)

    seconds = sum(int(row["duration_seconds"]) for row in rows)
    kinds: dict[str, int] = {}
    for row in rows:
        key = str(row["run_kind"])
        kinds[key] = kinds.get(key, 0) + 1
    print(f"Wrote {len(rows)} planned runs to {args.output}")
    print(
        "Run kinds: "
        + ", ".join(f"{key}={value}" for key, value in sorted(kinds.items()))
    )
    print(f"Declared collection time: {seconds / 3600:.1f} machine-hours")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


### `scripts/import_dice_baseline.py`


In [ ]:
%%writefile .prism_runtime/scripts/import_dice_baseline.py
#!/usr/bin/env python3
"""Import the frozen DICE M2 Pro baseline without bloating PRISM Git history."""

from __future__ import annotations

import argparse
import hashlib
import json
import shutil
import subprocess
from datetime import date
from pathlib import Path


SOURCE_REL = Path("data generation/dataset/ITC_M2Pro_DATA")
RAW_ITEMS = ("tier0", "tier1_alt", "tier2", "no_nan_report.json")
SUMMARY_FILES = (
    "RESULTS_SUMMARY.md",
    "case_inventory.csv",
    "case_predictions.csv",
    "config_runtime_summary.csv",
    "overall_metrics.csv",
    "run_context.json",
    "sequential_metrics.csv",
)


def git_value(repo: Path, *args: str) -> str:
    return subprocess.check_output(
        ("git", "-C", str(repo), *args), text=True
    ).strip()


def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def copy_item(source: Path, destination: Path) -> None:
    if source.is_dir():
        shutil.copytree(source, destination, dirs_exist_ok=True, copy_function=shutil.copy2)
    else:
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source, destination)


def copy_summaries(source_root: Path, destination_root: Path) -> list[str]:
    copied: list[str] = []
    profiles = {
        "mixed": ("results_dice_full", "results_dice_full_holdout"),
        "full": ("results_dice_full_full", "results_dice_full_full_holdout"),
    }
    for profile, (headline_dir, holdout_dir) in profiles.items():
        target = destination_root / profile
        target.mkdir(parents=True, exist_ok=True)
        for filename in SUMMARY_FILES:
            source = source_root / headline_dir / filename
            if not source.is_file():
                raise FileNotFoundError(source)
            shutil.copy2(source, target / filename)
            copied.append(str((target / filename).relative_to(destination_root)))
        holdout = source_root / holdout_dir / "holdout_robustness_summary.csv"
        if not holdout.is_file():
            raise FileNotFoundError(holdout)
        shutil.copy2(holdout, target / "holdout_robustness_summary.csv")
        copied.append(str((target / "holdout_robustness_summary.csv").relative_to(destination_root)))
    return copied


def main() -> int:
    parser = argparse.ArgumentParser()
    parser.add_argument(
        "--dice-root",
        type=Path,
        default=Path("../DICE"),
        help="Path to the local DICE checkout (default: ../DICE)",
    )
    parser.add_argument(
        "--prism-root",
        type=Path,
        default=Path(__file__).resolve().parents[1],
        help="PRISM repository root",
    )
    args = parser.parse_args()

    dice_root = args.dice_root.resolve()
    prism_root = args.prism_root.resolve()
    source_root = dice_root / SOURCE_REL
    payload_root = prism_root / "data/external/dice_m2pro_itc/payload"
    metadata_root = payload_root.parent
    baseline_root = prism_root / "data/baselines/dice_m2pro_itc"

    if not source_root.is_dir():
        raise SystemExit(f"DICE dataset not found: {source_root}")

    for item in RAW_ITEMS:
        source = source_root / item
        if not source.exists():
            raise SystemExit(f"Required DICE baseline item missing: {source}")
        copy_item(source, payload_root / item)

    copied_summaries = copy_summaries(source_root, baseline_root)

    files = sorted(path for path in payload_root.rglob("*") if path.is_file())
    checksum_lines = [
        f"{sha256(path)}  {path.relative_to(payload_root).as_posix()}" for path in files
    ]
    (metadata_root / "local-files.sha256").write_text(
        "\n".join(checksum_lines) + "\n", encoding="utf-8"
    )

    total_bytes = sum(path.stat().st_size for path in files)
    source_record = {
        "dataset_id": "dice_m2pro_itc",
        "role": "historical_baseline_only",
        "source_remote": git_value(dice_root, "remote", "get-url", "origin"),
        "source_commit": git_value(dice_root, "rev-parse", "HEAD"),
        "source_path": SOURCE_REL.as_posix(),
        "import_date": date.today().isoformat(),
        "payload_file_count": len(files),
        "payload_bytes": total_bytes,
        "payload_sha256_manifest": "local-files.sha256",
        "copied_summary_files": copied_summaries,
        "limitations": [
            "Apple M2 Pro/macOS only",
            "24 workload-scenario cases",
            "one recorded execution per case",
            "not independent PRISM replication",
        ],
    }
    (metadata_root / "source.json").write_text(
        json.dumps(source_record, indent=2) + "\n", encoding="utf-8"
    )

    print(f"Imported {len(files)} payload files ({total_bytes / 1024**2:.1f} MiB)")
    print(f"Source commit: {source_record['source_commit']}")
    print(f"Raw payload: {payload_root} (Git-ignored)")
    print(f"Tracked summaries: {baseline_root}")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


### `scripts/probe_collection.py`


In [ ]:
%%writefile .prism_runtime/scripts/probe_collection.py
#!/usr/bin/env python3
"""Report collection capabilities without recording a research run."""

from __future__ import annotations

import json
import platform
import shutil
import subprocess
from pathlib import Path

from prism_slm.collection import LinuxSysfsSampler, sanitized_platform_snapshot


def command_status(command: list[str]) -> dict[str, object]:
    executable = shutil.which(command[0])
    if executable is None:
        return {"available": False, "executable": None}
    try:
        result = subprocess.run(
            command,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            timeout=20,
            check=False,
        )
        return {
            "available": result.returncode == 0,
            "executable": executable,
            "return_code": result.returncode,
            "first_output_line": (result.stdout or "").strip().splitlines()[:1],
        }
    except subprocess.TimeoutExpired:
        return {"available": False, "executable": executable, "error": "timeout"}


def main() -> int:
    root = Path(__file__).resolve().parents[1]
    system = platform.system()
    platform_id = "M2_MACOS" if system == "Darwin" else "EPYC_LINUX"
    result: dict[str, object] = {
        "platform": sanitized_platform_snapshot(platform_id),
        "python": command_status(["python3", "--version"]),
    }
    if system == "Darwin":
        result["macmon"] = command_status(["macmon", "pipe", "-s", "1", "-i", "200"])
        result["xctrace"] = command_status(["xcrun", "xctrace", "version"])
    elif system == "Linux":
        sysfs = LinuxSysfsSampler()
        result["perf"] = command_status(["perf", "--version"])
        result["sensors"] = command_status(["sensors", "--version"])
        result["linux_sysfs"] = {
            "sensor_channels": sysfs.metadata(),
            "frequency_channel_count": len(sysfs.frequency_paths),
        }
        paranoid = Path("/proc/sys/kernel/perf_event_paranoid")
        result["perf_event_paranoid"] = (
            paranoid.read_text().strip() if paranoid.is_file() else None
        )

    destination = root / "data/collection-probes" / f"{platform_id.lower()}.json"
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_text(json.dumps(result, indent=2, sort_keys=True) + "\n")
    print(json.dumps(result, indent=2, sort_keys=True))
    print(f"Probe saved locally: {destination}")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


### `scripts/collect_run.py`


In [ ]:
%%writefile .prism_runtime/scripts/collect_run.py
#!/usr/bin/env python3
"""Collect one synchronized PRISM run."""

from __future__ import annotations

import argparse
from pathlib import Path

from prism_slm.collection import CollectionRequest, collect


def main() -> int:
    root = Path(__file__).resolve().parents[1]
    parser = argparse.ArgumentParser()
    parser.add_argument("--platform-id", required=True)
    parser.add_argument("--workload", required=True)
    parser.add_argument("--scenario", required=True)
    parser.add_argument("--repetition", type=int, required=True)
    parser.add_argument(
        "--split",
        choices=("calibration", "development", "locked_test", "smoke"),
        default="smoke",
    )
    parser.add_argument("--duration-seconds", type=int, default=30)
    parser.add_argument("--sampling-hz", type=float, default=5.0)
    parser.add_argument("--warmup-seconds", type=float, default=5.0)
    parser.add_argument(
        "--interruption-seconds",
        type=float,
        default=10.0,
        help="Masked enriched-telemetry interval for TELEMETRY_INTERRUPTION",
    )
    parser.add_argument("--purpose", choices=("smoke", "production"), default="smoke")
    parser.add_argument("--profile", choices=("portable", "enriched"), default="enriched")
    parser.add_argument("--run-id")
    parser.add_argument("--macmon-executable", default="macmon")
    parser.add_argument("--enable-xctrace", action="store_true")
    parser.add_argument("--xctrace-template", default="Time Profiler")
    parser.add_argument("--output-root", type=Path, default=root / "data/raw")
    args = parser.parse_args()

    run_dir = collect(
        CollectionRequest(
            platform_id=args.platform_id,
            workload=args.workload,
            scenario=args.scenario,
            repetition=args.repetition,
            split=args.split,
            duration_seconds=args.duration_seconds,
            sampling_hz=args.sampling_hz,
            warmup_seconds=args.warmup_seconds,
            interruption_seconds=args.interruption_seconds,
            purpose=args.purpose,
            profile=args.profile,
            output_root=args.output_root,
            run_id=args.run_id,
            macmon_executable=args.macmon_executable,
            enable_xctrace=args.enable_xctrace,
            xctrace_template=args.xctrace_template,
        )
    )
    print(f"Valid PRISM run: {run_dir}")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


### `scripts/validate_run.py`


In [ ]:
%%writefile .prism_runtime/scripts/validate_run.py
#!/usr/bin/env python3
"""Validate a completed PRISM run and refresh validation.json."""

from __future__ import annotations

import argparse
import json
from pathlib import Path

from prism_slm.collection import dump_json, validate_run


def main() -> int:
    parser = argparse.ArgumentParser()
    parser.add_argument("run_dir", type=Path)
    args = parser.parse_args()
    result = validate_run(args.run_dir, verify_hashes=True)
    dump_json(args.run_dir / "validation.json", result)
    print(json.dumps(result, indent=2))
    return 0 if result["valid"] else 1


if __name__ == "__main__":
    raise SystemExit(main())


### `scripts/check_collection_readiness.py`


In [ ]:
%%writefile .prism_runtime/scripts/check_collection_readiness.py
#!/usr/bin/env python3
"""Gate production collection on a clean revision and current smoke evidence."""

from __future__ import annotations

import argparse
import json
import platform
import shutil
import subprocess
import sys
from pathlib import Path

from prism_slm.collection import repository_state


def main() -> int:
    root = Path(__file__).resolve().parents[1]
    parser = argparse.ArgumentParser()
    parser.add_argument(
        "--platform-id",
        choices=("M2_MACOS", "EPYC_LINUX"),
        required=True,
    )
    parser.add_argument("--raw-root", type=Path, default=root / "data/raw")
    args = parser.parse_args()

    failures: list[str] = []
    expected_system = {"M2_MACOS": "Darwin", "EPYC_LINUX": "Linux"}[
        args.platform_id
    ]
    if platform.system() != expected_system:
        failures.append(
            f"{args.platform_id} requires {expected_system}, not {platform.system()}"
        )

    state = repository_state()
    if state.get("git_dirty") is not False:
        failures.append("Git checkout is not clean")
    commit = state.get("git_commit")

    preflight = subprocess.run(
        [
            sys.executable,
            str(root / "scripts/preflight.py"),
            str(root / "configs/experiment-matrix.toml"),
        ],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        check=False,
    )
    if preflight.returncode != 0:
        failures.append("experiment preflight failed")

    if args.platform_id == "M2_MACOS" and shutil.which("macmon") is None:
        failures.append("macmon is not installed or not on PATH")

    current_smoke: dict[str, Path] = {}
    platform_root = args.raw_root / args.platform_id
    for collection_path in sorted(
        platform_root.glob("*/*/collection.json"),
        reverse=True,
    ):
        run_dir = collection_path.parent
        validation_path = run_dir / "validation.json"
        if not validation_path.is_file():
            continue
        collection = json.loads(collection_path.read_text())
        validation = json.loads(validation_path.read_text())
        scenario = str(collection.get("scenario", ""))
        if (
            collection.get("purpose") == "smoke"
            and collection.get("platform_id") == args.platform_id
            and collection.get("profile") == "enriched"
            and collection.get("software", {}).get("git_commit") == commit
            and collection.get("software", {}).get("git_dirty") is False
            and validation.get("valid") is True
            and scenario in {"NOMINAL", "ATOMIC"}
            and scenario not in current_smoke
        ):
            current_smoke[scenario] = run_dir

    for scenario in ("NOMINAL", "ATOMIC"):
        if scenario not in current_smoke:
            failures.append(
                f"no valid clean-revision {scenario} smoke run for commit {commit}"
            )

    print(preflight.stdout.rstrip())
    print(f"repository commit: {commit}")
    for scenario, run_dir in sorted(current_smoke.items()):
        print(f"{scenario.lower()} smoke: {run_dir}")
    if failures:
        for failure in failures:
            print(f"ERROR: {failure}")
        return 1
    print(f"{args.platform_id} is ready for predeclared production collection")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


### `scripts/summarize_smoke.py`


In [ ]:
%%writefile .prism_runtime/scripts/summarize_smoke.py
#!/usr/bin/env python3
"""Compare a valid nominal/anomalous PRISM smoke pair."""

from __future__ import annotations

import argparse
import json
import statistics
from pathlib import Path
from typing import Any


METRICS = {
    "host_cpu_percent": ("host", "cpu_percent"),
    "host_cpu_utilization_derived_percent": (
        "host",
        "cpu_utilization_derived_percent",
    ),
    "cpu_power_w": ("enriched", "cpu_power"),
    "system_power_w": ("enriched", "sys_power"),
    "cpu_temperature_c": ("enriched", "temp", "cpu_temp_avg"),
}


def nested(record: dict[str, Any], path: tuple[str, ...]) -> float | None:
    value: Any = record
    for key in path:
        if not isinstance(value, dict) or key not in value:
            return None
        value = value[key]
    return float(value) if isinstance(value, (int, float)) else None


def rows(run_dir: Path) -> list[dict[str, Any]]:
    validation = json.loads((run_dir / "validation.json").read_text())
    if not validation.get("valid"):
        raise ValueError(f"run is not valid: {run_dir}")
    return [
        json.loads(line)
        for line in (run_dir / "telemetry.jsonl").read_text().splitlines()
        if line.strip()
    ]


def medians(records: list[dict[str, Any]]) -> dict[str, float | None]:
    result: dict[str, float | None] = {}
    for name, path in METRICS.items():
        values = [value for row in records if (value := nested(row, path)) is not None]
        result[name] = statistics.median(values) if values else None
    return result


def main() -> int:
    parser = argparse.ArgumentParser()
    parser.add_argument("nominal_run", type=Path)
    parser.add_argument("anomalous_run", type=Path)
    parser.add_argument("--output", type=Path)
    args = parser.parse_args()

    nominal = rows(args.nominal_run)
    anomalous = rows(args.anomalous_run)
    events = [
        json.loads(line)
        for line in (args.anomalous_run / "events.jsonl").read_text().splitlines()
        if line.strip()
    ]
    onset_events = [event for event in events if event.get("event") == "stressor_started"]
    if len(onset_events) != 1:
        raise ValueError("anomalous run must contain exactly one stressor_started event")
    onset = float(onset_events[0]["t_rel_seconds"])
    before = [row for row in anomalous if float(row["t_rel_seconds"]) < onset]
    after = [row for row in anomalous if float(row["t_rel_seconds"]) >= onset]

    summary = {
        "nominal_run": args.nominal_run.name,
        "anomalous_run": args.anomalous_run.name,
        "stressor_onset_seconds": onset,
        "sample_counts": {
            "nominal": len(nominal),
            "anomalous_pre_onset": len(before),
            "anomalous_post_onset": len(after),
        },
        "nominal_medians": medians(nominal),
        "anomalous_pre_onset_medians": medians(before),
        "anomalous_post_onset_medians": medians(after),
    }
    text = json.dumps(summary, indent=2, sort_keys=True) + "\n"
    if args.output:
        args.output.parent.mkdir(parents=True, exist_ok=True)
        args.output.write_text(text, encoding="utf-8")
    print(text, end="")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


### `scripts/collect_next.py`


In [ ]:
%%writefile .prism_runtime/scripts/collect_next.py
#!/usr/bin/env python3
"""Preview or execute the next predeclared PRISM production run."""

from __future__ import annotations

import argparse
import csv
import platform
import shlex
import shutil
from pathlib import Path

from prism_slm.collection import CollectionRequest, collect, update_collection_plan


def _default_platform() -> str:
    system = platform.system()
    if system == "Darwin":
        return "M2_MACOS"
    if system == "Linux":
        return "EPYC_LINUX"
    raise SystemExit(f"Unsupported collection operating system: {system}")


def _read_rows(path: Path) -> tuple[list[str], list[dict[str, str]]]:
    with path.open(newline="", encoding="utf-8") as stream:
        reader = csv.DictReader(stream)
        return list(reader.fieldnames or []), list(reader)


def _ensure_progress(plan_path: Path, progress_path: Path) -> list[dict[str, str]]:
    _, plan_rows = _read_rows(plan_path)
    if not progress_path.exists():
        progress_path.parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(plan_path, progress_path)
    _, progress_rows = _read_rows(progress_path)
    if [row["run_id"] for row in progress_rows] != [
        row["run_id"] for row in plan_rows
    ]:
        raise SystemExit(
            "Collection progress does not match the immutable plan. Preserve the "
            "progress file and reconcile it manually; do not overwrite collected state."
        )
    return progress_rows


def _matches(row: dict[str, str], args: argparse.Namespace) -> bool:
    if row["platform_id"] != args.platform_id:
        return False
    if row["status"] != "planned":
        return False
    for field in ("run_id", "run_kind", "workload", "scenario", "planned_split"):
        expected = getattr(args, field)
        if expected and row[field].upper() != expected.upper():
            return False
    if row["planned_split"] == "locked_test" and not args.unlock_locked_test:
        return False
    return True


def _command(row: dict[str, str]) -> list[str]:
    command = [
        "python3",
        "scripts/collect_run.py",
        "--platform-id",
        row["platform_id"],
        "--workload",
        row["workload"],
        "--scenario",
        row["scenario"],
        "--repetition",
        row["repetition"],
        "--split",
        row["planned_split"],
        "--purpose",
        row["purpose"],
        "--profile",
        row["profile"],
        "--run-id",
        row["run_id"],
        "--duration-seconds",
        row["duration_seconds"],
        "--sampling-hz",
        row["sampling_hz"],
        "--warmup-seconds",
        row["warmup_seconds"],
    ]
    if float(row["interruption_seconds"]) > 0:
        command.extend(["--interruption-seconds", row["interruption_seconds"]])
    return command


def main() -> int:
    root = Path(__file__).resolve().parents[1]
    parser = argparse.ArgumentParser()
    parser.add_argument("--platform-id", default=_default_platform())
    parser.add_argument("--run-id")
    parser.add_argument("--run-kind")
    parser.add_argument("--workload")
    parser.add_argument("--scenario")
    parser.add_argument("--planned-split")
    parser.add_argument(
        "--plan",
        type=Path,
        default=root / "data/collection-plan.csv",
    )
    parser.add_argument(
        "--progress",
        type=Path,
        default=root / "data/collection-progress.csv",
    )
    parser.add_argument("--output-root", type=Path, default=root / "data/raw")
    parser.add_argument(
        "--execute",
        action="store_true",
        help="Run the selected production cell; without this flag only preview it",
    )
    parser.add_argument(
        "--unlock-locked-test",
        action="store_true",
        help="Allow a predeclared locked-test row after the method is frozen",
    )
    args = parser.parse_args()

    rows = _ensure_progress(args.plan, args.progress)
    selected = next((row for row in rows if _matches(row, args)), None)
    if selected is None:
        raise SystemExit(
            "No matching planned run is available. Locked-test rows are hidden "
            "unless --unlock-locked-test is supplied."
        )

    print("Selected predeclared run:")
    for field in (
        "run_id",
        "platform_id",
        "run_kind",
        "workload",
        "scenario",
        "planned_split",
        "duration_seconds",
        "sampling_hz",
        "warmup_seconds",
        "interruption_seconds",
    ):
        print(f"  {field}: {selected[field]}")
    print("\nEquivalent low-level command (does not update progress):")
    print(shlex.join(_command(selected)))
    if not args.execute:
        print("\nPreview only. Re-run with --execute after reviewing the selection.")
        return 0

    run_dir = collect(
        CollectionRequest(
            platform_id=selected["platform_id"],
            workload=selected["workload"],
            scenario=selected["scenario"],
            repetition=int(selected["repetition"]),
            split=selected["planned_split"],
            duration_seconds=int(selected["duration_seconds"]),
            sampling_hz=float(selected["sampling_hz"]),
            warmup_seconds=float(selected["warmup_seconds"]),
            interruption_seconds=float(selected["interruption_seconds"]),
            purpose=selected["purpose"],
            profile=selected["profile"],
            output_root=args.output_root,
            run_id=selected["run_id"],
        )
    )
    update_collection_plan(args.progress, run_dir)
    print(f"Valid PRISM run: {run_dir}")
    print(f"Progress updated: {args.progress}")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


### `scripts/collection_status.py`


In [ ]:
%%writefile .prism_runtime/scripts/collection_status.py
#!/usr/bin/env python3
"""Summarize immutable plan coverage and local collection progress."""

from __future__ import annotations

import argparse
import csv
from collections import Counter, defaultdict
from pathlib import Path


def main() -> int:
    root = Path(__file__).resolve().parents[1]
    parser = argparse.ArgumentParser()
    parser.add_argument(
        "--progress",
        type=Path,
        default=root / "data/collection-progress.csv",
    )
    parser.add_argument(
        "--plan",
        type=Path,
        default=root / "data/collection-plan.csv",
    )
    args = parser.parse_args()
    source = args.progress if args.progress.exists() else args.plan
    with source.open(newline="", encoding="utf-8") as stream:
        rows = list(csv.DictReader(stream))

    by_platform: dict[str, Counter[str]] = defaultdict(Counter)
    benign_hours: dict[str, float] = defaultdict(float)
    for row in rows:
        by_platform[row["platform_id"]][row["status"]] += 1
        if row["status"] == "valid" and row["scenario"] == "NOMINAL":
            benign_hours[row["platform_id"]] += float(row["benign_hours"] or 0)

    print(f"Collection tracker: {source}")
    for platform_id in sorted(by_platform):
        statuses = ", ".join(
            f"{name}={count}" for name, count in sorted(by_platform[platform_id].items())
        )
        print(
            f"{platform_id}: {statuses}; valid benign hours={benign_hours[platform_id]:.2f}"
        )
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


### `scripts/data_quality_report.py`


In [ ]:
%%writefile .prism_runtime/scripts/data_quality_report.py
#!/usr/bin/env python3
"""Aggregate per-run validation into a platform data-quality report."""

from __future__ import annotations

import argparse
import json
import statistics
from collections import Counter, defaultdict
from pathlib import Path


def main() -> int:
    root = Path(__file__).resolve().parents[1]
    parser = argparse.ArgumentParser()
    parser.add_argument("--platform-id", required=True)
    parser.add_argument("--raw-root", type=Path, default=root / "data/raw")
    parser.add_argument(
        "--output",
        type=Path,
        default=None,
    )
    parser.add_argument("--include-smoke", action="store_true")
    args = parser.parse_args()

    runs: list[dict] = []
    for validation_path in sorted(
        (args.raw_root / args.platform_id).glob("*/*/validation.json")
    ):
        run_dir = validation_path.parent
        collection_path = run_dir / "collection.json"
        if not collection_path.is_file():
            continue
        collection = json.loads(collection_path.read_text())
        if not args.include_smoke and collection.get("purpose") != "production":
            continue
        validation = json.loads(validation_path.read_text())
        runs.append(
            {
                "run_id": collection.get("run_id"),
                "scenario": collection.get("scenario"),
                "workload": collection.get("workload"),
                "purpose": collection.get("purpose"),
                "valid": validation.get("valid", False),
                "duration_observed_seconds": validation.get(
                    "duration_observed_seconds",
                    0,
                ),
                "warnings": validation.get("warnings", []),
                "failures": validation.get("failures", []),
                "quality": validation.get("quality", {}),
            }
        )

    statuses = Counter("valid" if run["valid"] else "invalid" for run in runs)
    scenario_counts: dict[str, Counter[str]] = defaultdict(Counter)
    channel_fractions: dict[str, list[float]] = defaultdict(list)
    warning_counts: Counter[str] = Counter()
    failure_counts: Counter[str] = Counter()
    benign_hours = 0.0
    for run in runs:
        status = "valid" if run["valid"] else "invalid"
        scenario_counts[str(run["scenario"])][status] += 1
        if run["valid"] and run["scenario"] == "NOMINAL":
            benign_hours += float(run["duration_observed_seconds"]) / 3600
        for warning in run["warnings"]:
            warning_counts[str(warning)] += 1
        for failure in run["failures"]:
            failure_counts[str(failure)] += 1
        for name, metrics in run["quality"].get("critical_channels", {}).items():
            channel_fractions[name].append(float(metrics["non_null_fraction"]))

    report = {
        "platform_id": args.platform_id,
        "run_count": len(runs),
        "statuses": dict(sorted(statuses.items())),
        "valid_benign_hours": benign_hours,
        "scenario_counts": {
            name: dict(sorted(counts.items()))
            for name, counts in sorted(scenario_counts.items())
        },
        "critical_channel_availability": {
            name: {
                "minimum": min(values),
                "median": statistics.median(values),
                "maximum": max(values),
                "run_count": len(values),
            }
            for name, values in sorted(channel_fractions.items())
            if values
        },
        "warning_counts": dict(sorted(warning_counts.items())),
        "failure_counts": dict(sorted(failure_counts.items())),
    }
    destination = args.output or (
        root / "artifacts/data-quality" / f"{args.platform_id.lower()}.json"
    )
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_text(json.dumps(report, indent=2, sort_keys=True) + "\n")
    print(json.dumps(report, indent=2, sort_keys=True))
    print(f"Quality report: {destination}")
    return 0 if statuses.get("invalid", 0) == 0 else 1


if __name__ == "__main__":
    raise SystemExit(main())


### `tests/test_contracts.py`


In [ ]:
%%writefile .prism_runtime/tests/test_contracts.py
import unittest

from prism_slm.contracts import ChannelSpec, PlatformSpec, SemanticGroup


class ContractTests(unittest.TestCase):
    def test_platform_reports_available_semantic_groups(self) -> None:
        platform = PlatformSpec.from_channels(
            platform_id="EPYC_LINUX",
            architecture="x86_64",
            operating_system="Ubuntu",
            observability="host",
            channels=[
                ChannelSpec(
                    native_name="cpu_utilization",
                    unit="percent",
                    semantic_group=SemanticGroup.COMPUTE,
                    source="procfs",
                    sampling_period_seconds=1.0,
                ),
                ChannelSpec(
                    native_name="memory_available",
                    unit="bytes",
                    semantic_group=SemanticGroup.MEMORY,
                    source="procfs",
                    sampling_period_seconds=1.0,
                ),
            ],
        )
        self.assertEqual(
            platform.available_groups,
            {SemanticGroup.COMPUTE, SemanticGroup.MEMORY},
        )

    def test_duplicate_native_channels_are_rejected(self) -> None:
        channel = ChannelSpec(
            native_name="temperature",
            unit="celsius",
            semantic_group=SemanticGroup.THERMAL_POWER,
            source="lm_sensors",
            sampling_period_seconds=1.0,
        )
        with self.assertRaisesRegex(ValueError, "unique"):
            PlatformSpec.from_channels(
                platform_id="EPYC_LINUX",
                architecture="x86_64",
                operating_system="Ubuntu",
                observability="host",
                channels=[channel, channel],
            )

    def test_nonpositive_sampling_period_is_rejected(self) -> None:
        with self.assertRaisesRegex(ValueError, "positive"):
            ChannelSpec(
                native_name="power",
                unit="watts",
                semantic_group=SemanticGroup.THERMAL_POWER,
                source="collector",
                sampling_period_seconds=0,
            )


if __name__ == "__main__":
    unittest.main()


### `tests/test_collection.py`


In [ ]:
%%writefile .prism_runtime/tests/test_collection.py
import tempfile
import unittest
import json
from pathlib import Path

from prism_slm.collection import (
    channel_group,
    channel_unit,
    flatten_leaves,
    sanitize_enriched_sample,
    telemetry_quality_summary,
    validate_run,
    verify_checksums,
    write_checksums,
)
from prism_slm.contracts import SemanticGroup


class CollectionTests(unittest.TestCase):
    def test_channel_semantics(self) -> None:
        self.assertEqual(
            channel_group("enriched.temp.cpu_temp_avg"),
            SemanticGroup.THERMAL_POWER,
        )
        self.assertEqual(
            channel_group("enriched.gpu_power"),
            SemanticGroup.ACCELERATOR,
        )
        self.assertEqual(channel_group("host.memory.available"), SemanticGroup.MEMORY)
        self.assertEqual(channel_group("host.network.bytes_sent_total"), SemanticGroup.IO)

    def test_channel_units(self) -> None:
        self.assertEqual(channel_unit("enriched.cpu_power"), "watts")
        self.assertEqual(channel_unit("enriched.temp.cpu_temp_avg"), "celsius")
        self.assertEqual(channel_unit("host.memory.available"), "bytes")
        self.assertEqual(channel_unit("enriched.pcpu_usage.0"), "megahertz")
        self.assertEqual(channel_unit("enriched.pcpu_usage.1"), "fraction")
        self.assertEqual(
            channel_unit("host.cpu_utilization_derived_percent"),
            "percent",
        )

    def test_flatten_and_checksum(self) -> None:
        self.assertEqual(
            flatten_leaves({"cpu": {"usage": [1000, 0.5]}}),
            {"cpu.usage.0": 1000, "cpu.usage.1": 0.5},
        )
        with tempfile.TemporaryDirectory() as directory:
            root = Path(directory)
            (root / "telemetry.jsonl").write_text('{"sample": 1}\n')
            write_checksums(root)
            self.assertEqual(verify_checksums(root), [])
            (root / "telemetry.jsonl").write_text('{"sample": 2}\n')
            self.assertIn("checksum mismatch", verify_checksums(root)[0])

    def test_impossible_apple_temperature_is_nulled_but_input_is_preserved(self) -> None:
        raw = {
            "cpu_power": 8.0,
            "sys_power": 18.0,
            "pcpu_usage": [2400, 0.25],
            "temp": {"cpu_temp_avg": 9.8},
        }
        sanitized, flags = sanitize_enriched_sample("M2_MACOS", raw)
        self.assertEqual(raw["temp"]["cpu_temp_avg"], 9.8)
        self.assertIsNone(sanitized["temp"]["cpu_temp_avg"])
        self.assertIn(
            "out_of_range:enriched.temp.cpu_temp_avg",
            flags,
        )

    def test_quality_summary_excludes_declared_interruption(self) -> None:
        samples = [
            {
                "enriched": {
                    "cpu_power": 8.0,
                    "sys_power": 18.0,
                    "pcpu_usage": [2400, 0.25],
                    "temp": {"cpu_temp_avg": 45.0},
                },
                "availability": {"interruption_injected": False},
            },
            {
                "enriched": None,
                "availability": {"interruption_injected": True},
            },
        ]
        summary = telemetry_quality_summary(samples, "M2_MACOS")
        self.assertEqual(summary["interruption_masked_samples"], 1)
        self.assertEqual(
            summary["critical_channels"]["enriched.cpu_power"]["non_null_fraction"],
            1.0,
        )

    def test_validator_rejects_compressed_timeline(self) -> None:
        with tempfile.TemporaryDirectory() as directory:
            root = Path(directory)
            (root / "platform.json").write_text("{}\n")
            (root / "channels.json").write_text('{"channels": []}\n')
            (root / "collection.json").write_text(
                json.dumps(
                    {
                        "expected_samples": 3,
                        "sampling_hz": 1,
                        "scenario": "NOMINAL",
                        "status": "complete",
                    }
                )
                + "\n"
            )
            (root / "events.jsonl").write_text(
                '{"event": "workload_started"}\n'
            )
            (root / "telemetry.jsonl").write_text(
                "\n".join(
                    json.dumps(
                        {
                            "t_rel_seconds": value,
                            "enriched": None,
                        }
                    )
                    for value in (0.0, 0.01, 0.02)
                )
                + "\n"
            )
            result = validate_run(root, verify_hashes=False)
            self.assertFalse(result["valid"])
            self.assertTrue(
                any("observed sample span" in item for item in result["failures"])
            )

    def test_validator_accepts_controlled_crash_event(self) -> None:
        with tempfile.TemporaryDirectory() as directory:
            root = Path(directory)
            (root / "platform.json").write_text("{}\n")
            (root / "channels.json").write_text('{"channels": []}\n')
            (root / "collection.json").write_text(
                json.dumps(
                    {
                        "expected_samples": 3,
                        "sampling_hz": 1,
                        "scenario": "CONTROLLED_CRASH",
                        "status": "complete",
                    }
                )
                + "\n"
            )
            (root / "events.jsonl").write_text(
                '{"event": "workload_started"}\n'
                '{"event": "controlled_crash_injected"}\n'
            )
            (root / "telemetry.jsonl").write_text(
                "\n".join(
                    json.dumps(
                        {
                            "t_rel_seconds": value,
                            "enriched": None,
                        }
                    )
                    for value in (0.0, 1.0, 2.0)
                )
                + "\n"
            )
            result = validate_run(root, verify_hashes=False)
            self.assertTrue(result["valid"], result["failures"])


if __name__ == "__main__":
    unittest.main()


### `tests/test_collection_plan.py`


In [ ]:
%%writefile .prism_runtime/tests/test_collection_plan.py
import importlib.util
import tomllib
import unittest
from collections import Counter
from pathlib import Path

from prism_slm.workload_harness import SCENARIOS


ROOT = Path(__file__).resolve().parents[1]
SPEC = importlib.util.spec_from_file_location(
    "make_collection_plan",
    ROOT / "scripts/make_collection_plan.py",
)
assert SPEC is not None and SPEC.loader is not None
MODULE = importlib.util.module_from_spec(SPEC)
SPEC.loader.exec_module(MODULE)


class CollectionPlanTests(unittest.TestCase):
    @classmethod
    def setUpClass(cls) -> None:
        with (ROOT / "configs/experiment-matrix.toml").open("rb") as stream:
            cls.config = tomllib.load(stream)
        cls.rows = MODULE.build_rows(cls.config)

    def test_extension_plan_inventory(self) -> None:
        self.assertEqual(len(self.rows), 252)
        self.assertEqual(
            Counter(row["run_kind"] for row in self.rows),
            {
                "required_matrix": 192,
                "targeted_extension": 36,
                "long_benign": 24,
            },
        )
        total_hours = sum(int(row["duration_seconds"]) for row in self.rows) / 3600
        self.assertAlmostEqual(total_hours, 64.8)

    def test_each_platform_has_twelve_planned_benign_hours(self) -> None:
        for platform_id in ("M2_MACOS", "EPYC_LINUX"):
            hours = (
                sum(
                    int(row["duration_seconds"])
                    for row in self.rows
                    if row["platform_id"] == platform_id
                    and row["scenario"] == "NOMINAL"
                )
                / 3600
            )
            self.assertAlmostEqual(hours, 12.0)

    def test_targeted_extension_scenarios_are_repeated(self) -> None:
        targeted = {
            row["scenario"]
            for row in self.rows
            if row["run_kind"] == "targeted_extension"
        }
        self.assertEqual(
            targeted,
            {"THERMAL_SHIFT", "POWER_SHIFT", "DEGRADATION_PROXY"},
        )
        planned_scenarios = {row["scenario"] for row in self.rows}
        self.assertTrue(planned_scenarios.issubset(set(SCENARIOS)))


if __name__ == "__main__":
    unittest.main()


## 3. Wire repository data/configuration into the runtime


In [ ]:

shutil.copy2(REPO_ROOT / "pyproject.toml", RUNTIME_ROOT / "pyproject.toml")

for directory_name in ("configs", "data", "docs", "paper"):
    link = RUNTIME_ROOT / directory_name
    destination = REPO_ROOT / directory_name
    if link.is_symlink():
        if link.resolve() != destination.resolve():
            link.unlink()
        else:
            continue
    if link.exists():
        raise RuntimeError(
            f"{link} exists but is not the notebook-managed symlink. "
            "Move it aside, then rerun this cell."
        )
    link.symlink_to(destination, target_is_directory=True)

print("Runtime source and repository data/configuration are connected.")


## 4. Shared notebook helpers


In [ ]:

def run_script(
    script_name: str,
    *arguments: object,
    check: bool = True,
) -> subprocess.CompletedProcess[str]:
    command = [
        sys.executable,
        str(RUNTIME_ROOT / "scripts" / script_name),
        *(str(argument) for argument in arguments),
    ]
    environment = os.environ.copy()
    environment["PRISM_REPO_ROOT"] = str(REPO_ROOT)
    environment["PYTHONPATH"] = os.environ["PYTHONPATH"]
    print("$", " ".join(command))
    return subprocess.run(
        command,
        cwd=REPO_ROOT,
        env=environment,
        check=check,
        text=True,
    )


def run_tests() -> subprocess.CompletedProcess[str]:
    command = [
        sys.executable,
        "-m",
        "unittest",
        "discover",
        "-s",
        str(RUNTIME_ROOT / "tests"),
        "-v",
    ]
    print("$", " ".join(command))
    return subprocess.run(
        command,
        cwd=RUNTIME_ROOT,
        env=os.environ.copy(),
        check=True,
        text=True,
    )


## 5. Automatic integrity gates


In [ ]:

run_script("preflight.py", REPO_ROOT / "configs" / "experiment-matrix.toml")
run_tests()
print("PASS: experiment configuration and embedded source tests are valid.")


## 6. Frozen DICE baseline import (optional)

DICE is retained only for conference-baseline reproduction and retrospective
sequential-scoring comparisons. It does not count as new PRISM replication.
Set the path to your local DICE checkout and change the guard only when needed.


In [ ]:

RUN_DICE_IMPORT = False
DICE_ROOT = REPO_ROOT.parent / "DICE"

if RUN_DICE_IMPORT:
    run_script("import_dice_baseline.py", "--dice-root", DICE_ROOT)
else:
    print("Skipped DICE import. Set RUN_DICE_IMPORT=True to enable it.")


## 7. Platform capability probe (Apple first, then EPYC/Linux)

Run this notebook on the target machine in a normal terminal-launched Jupyter
session. The same cell detects the current platform and records a local,
Git-ignored capability report.


In [ ]:

RUN_CAPABILITY_PROBE = False

if RUN_CAPABILITY_PROBE:
    run_script("probe_collection.py")
else:
    print("Skipped probe. Set RUN_CAPABILITY_PROBE=True on the target machine.")


## 8. Required Apple M2 Pro smoke pair

Requirements: Apple Silicon macOS, `macmon`, and an idle machine. The anomalous
run has a 10-second clean pre-onset window. Each run is bounded to 30 seconds.


In [ ]:

RUN_APPLE_SMOKE_PAIR = False

if RUN_APPLE_SMOKE_PAIR:
    common = (
        "--platform-id", "M2_MACOS",
        "--workload", "PY_STATS",
        "--repetition", 0,
        "--split", "smoke",
        "--purpose", "smoke",
        "--profile", "enriched",
        "--duration-seconds", 30,
        "--sampling-hz", 5,
    )
    run_script("collect_run.py", *common, "--scenario", "NOMINAL")
    run_script(
        "collect_run.py",
        *common,
        "--scenario", "ATOMIC",
        "--warmup-seconds", 10,
    )
    run_script("check_collection_readiness.py", "--platform-id", "M2_MACOS")
else:
    print("Skipped Apple smoke pair. Set RUN_APPLE_SMOKE_PAIR=True on the M2.")


## 9. Required AMD EPYC/Linux smoke pair

Run only inside an authorized compute allocation, never on a shared login node.
Missing enriched sensors are retained as explicit availability results.


In [ ]:

RUN_LINUX_SMOKE_PAIR = False

if RUN_LINUX_SMOKE_PAIR:
    common = (
        "--platform-id", "EPYC_LINUX",
        "--workload", "PY_STATS",
        "--repetition", 0,
        "--split", "smoke",
        "--purpose", "smoke",
        "--profile", "enriched",
        "--duration-seconds", 30,
        "--sampling-hz", 5,
    )
    run_script("collect_run.py", *common, "--scenario", "NOMINAL")
    run_script(
        "collect_run.py",
        *common,
        "--scenario", "ATOMIC",
        "--warmup-seconds", 10,
    )
    run_script("check_collection_readiness.py", "--platform-id", "EPYC_LINUX")
else:
    print("Skipped Linux smoke pair. Set RUN_LINUX_SMOKE_PAIR=True on EPYC.")


## 10. Inspect or compare collected smoke runs


In [ ]:

VALIDATE_RUN = False
RUN_DIRECTORY = REPO_ROOT / "data" / "raw" / "<platform>/<date>/<run_id>"

if VALIDATE_RUN:
    run_script("validate_run.py", RUN_DIRECTORY)
else:
    print("Set RUN_DIRECTORY and VALIDATE_RUN=True to revalidate one run.")


In [ ]:

COMPARE_SMOKE_PAIR = False
NOMINAL_RUN = REPO_ROOT / "data" / "raw" / "<platform>/<date>/<nominal-run-id>"
ANOMALOUS_RUN = REPO_ROOT / "data" / "raw" / "<platform>/<date>/<atomic-run-id>"

if COMPARE_SMOKE_PAIR:
    run_script("summarize_smoke.py", NOMINAL_RUN, ANOMALOUS_RUN)
else:
    print("Set both run paths and COMPARE_SMOKE_PAIR=True to compare them.")


## 11. Preview the next predeclared production row

Preview is non-collecting. Change the platform when this notebook is running on
EPYC/Linux. Locked-test rows remain hidden until the method is frozen.


In [ ]:

PREVIEW_NEXT_RUN = False
PLATFORM_ID = "M2_MACOS"  # change to "EPYC_LINUX" on the ASU server
RUN_KIND = None           # e.g. "required_matrix" or "long_benign"
WORKLOAD = None           # e.g. "PY_STATS"
SCENARIO = None           # e.g. "NOMINAL"

if PREVIEW_NEXT_RUN:
    filters: list[object] = ["--platform-id", PLATFORM_ID]
    for flag, value in (
        ("--run-kind", RUN_KIND),
        ("--workload", WORKLOAD),
        ("--scenario", SCENARIO),
    ):
        if value:
            filters.extend((flag, value))
    run_script("collect_next.py", *filters)
else:
    print("Set PREVIEW_NEXT_RUN=True to inspect the next eligible plan row.")


## 12. Execute one production row (explicitly guarded)

This is the only production execution cell. It refuses dirty Git state and
requires matching smoke evidence from the exact commit. Review the preview
first. Do not unlock held-out test rows until the method-freeze decision has
been recorded.


In [ ]:

EXECUTE_PRODUCTION = False
UNLOCK_LOCKED_TEST = False

if EXECUTE_PRODUCTION:
    arguments: list[object] = ["--platform-id", PLATFORM_ID, "--execute"]
    if RUN_KIND:
        arguments.extend(("--run-kind", RUN_KIND))
    if WORKLOAD:
        arguments.extend(("--workload", WORKLOAD))
    if SCENARIO:
        arguments.extend(("--scenario", SCENARIO))
    if UNLOCK_LOCKED_TEST:
        arguments.append("--unlock-locked-test")
    run_script("collect_next.py", *arguments)
else:
    print(
        "Production collection is OFF. Preview first, then set "
        "EXECUTE_PRODUCTION=True for exactly one row."
    )


## 13. Daily progress and channel-quality gates


In [ ]:

RUN_DAILY_REPORT = False

if RUN_DAILY_REPORT:
    run_script("collection_status.py")
    run_script(
        "data_quality_report.py",
        "--platform-id", PLATFORM_ID,
        "--include-smoke",
    )
else:
    print("Set RUN_DAILY_REPORT=True at the end of each collection day.")


## 14. Completion criteria

Data collection is complete only when:

- all required-matrix and targeted-extension rows are valid or have documented
  replacement rows;
- Apple and EPYC each have at least 12 valid benign hours;
- telemetry missingness and quality reports are frozen by platform;
- calibration/development were completed before the detector/method freeze;
- locked-test traces were not used for tuning;
- raw files, manifests, and progress trackers have access-controlled backups.

After collection, the next notebook revision should add the DICE-vs-PRISM
detector analysis: fixed persistence, CUSUM/EWMA, and a conformal
test-martingale/e-process evaluated by false alerts per hour, time to detect,
platform-transfer performance, and calibration effort.
